File path :
[Databricks_notebook](https://dbc-6fd9c644-9fe4.cloud.databricks.com/editor/notebooks/2763060734768307?o=7474647103420595)

Notes:

Count -- is both action and transformation

depends on where we use it

df.count() --- it is a action

df.select(count('col_name'))---- it is a transformation

#### Load file In Spark 
(All important data": https://github.com/databricks/Spark-The-Definitive-Guide/tree/master/data)

##### Loading a CSV File

In [0]:
spark

In [0]:
flight_df = spark.read.format("csv")\
          .option("header", "false")\
          .option("inferschema", "false")\
          .option('mode','FAILFAST')\
          .load("/Volumes/workspace/default/data_files/2015-summary.csv")
flight_df.show(5)

+-----------------+-------------------+-----+
|              _c0|                _c1|  _c2|
+-----------------+-------------------+-----+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|
|    United States|            Romania|   15|
|    United States|            Croatia|    1|
|    United States|            Ireland|  344|
|            Egypt|      United States|   15|
+-----------------+-------------------+-----+
only showing top 5 rows


In [0]:
flight_df = spark.read.format("csv")\
          .option("header", "true")\
          .option("inferschema", "true")\
          .option('mode','FAILFAST')\
          .load("/Volumes/workspace/default/data_files/2015-summary.csv")
flight_df.show(5)

+-----------------+-------------------+-----+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|
+-----------------+-------------------+-----+
|    United States|            Romania|   15|
|    United States|            Croatia|    1|
|    United States|            Ireland|  344|
|            Egypt|      United States|   15|
|    United States|              India|   62|
+-----------------+-------------------+-----+
only showing top 5 rows


In [0]:
flight_df.explain()

== Physical Plan ==
FileScan csv [DEST_COUNTRY_NAME#11010,ORIGIN_COUNTRY_NAME#11011,count#11012] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[dbfs:/Volumes/workspace/default/data_files/2015-summary.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<DEST_COUNTRY_NAME:string,ORIGIN_COUNTRY_NAME:string,count:int>




In [0]:
flight_df.printSchema()

root
 |-- DEST_COUNTRY_NAME: string (nullable = true)
 |-- ORIGIN_COUNTRY_NAME: string (nullable = true)
 |-- count: integer (nullable = true)



###### Creating Manual Schema

In [0]:
%fs
ls /Volumes/workspace/default/data_files

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/data_files/2015-summary.csv,2015-summary.csv,7080,1775730449000
dbfs:/Volumes/workspace/default/data_files/Employee_data.csv,Employee_data.csv,230,1776145590000
dbfs:/Volumes/workspace/default/data_files/Multi_line_correct.json,Multi_line_correct.json,310,1776165645000
dbfs:/Volumes/workspace/default/data_files/Multi_line_incorrect.json,Multi_line_incorrect.json,304,1776165645000
dbfs:/Volumes/workspace/default/data_files/corrupted_data/,corrupted_data/,0,1779868422668
dbfs:/Volumes/workspace/default/data_files/corrupted_json.json,corrupted_json.json,218,1776165414000
dbfs:/Volumes/workspace/default/data_files/file5.json,file5.json,669503,1776229378000
dbfs:/Volumes/workspace/default/data_files/line_delimited_json.json,line_delimited_json.json,219,1776163722000
dbfs:/Volumes/workspace/default/data_files/part-r-00000-1a9822ba-b8fb-4d8e-844a-ea30d0801b9e.gz.parquet,part-r-00000-1a9822ba-b8fb-4d8e-844a-ea30d0801b9e.gz.parquet,3921,1776328873000
dbfs:/Volumes/workspace/default/data_files/single_file_json.json,single_file_json.json,232,1776164511000


In [0]:
flight_df = spark.read.format("csv")\
          .option("header", "true")\
          .option("inferschema", "false")\
          .schema(myschema)\
          .option('mode','FAILFAST')\
          .load("/Volumes/workspace/default/data_files/2015-summary.csv")
flight_df.show(5)

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-4630086708793936>, line 4
      1 flight_df = spark.read.format("csv")\
      2           .option("header", "true")\
      3           .option("inferschema", "false")\
----> 4           .schema(myschema)\
      5           .option('mode','FAILFAST')\
      6           .load("/Volumes/workspace/default/data_files/2015-summary.csv")
      7 flight_df.show(5)

NameError: name 'myschema' is not defined

In [0]:
from pyspark.sql.types import StructType,StructField,StringType,IntegerType

my_schema = StructType([
            StructField("DEST_COUNTRY_NAME", StringType(), True), # Column_name, Datatype, Nullable
            StructField("ORIGIN_COUNTRY_NAME", StringType(), True),
            StructField("count", IntegerType(), True)
            
])


flight_df = spark.read.format("csv")\
          .option("header", "true")\
          .option("inferschema", "false")\
          .schema(my_schema)\
          .option('mode','FAILFAST')\
          .load("/Volumes/workspace/default/data_files/2015-summary.csv")
flight_df.show(5)

+-----------------+-------------------+-----+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|
+-----------------+-------------------+-----+
|    United States|            Romania|   15|
|    United States|            Croatia|    1|
|    United States|            Ireland|  344|
|            Egypt|      United States|   15|
|    United States|              India|   62|
+-----------------+-------------------+-----+
only showing top 5 rows


###### Standard PySpark does not use SkipRows. If we have header set to true

In [0]:
from pyspark.sql.types import StructType,StructField,StringType,IntegerType

my_schema_2 = StructType([
            StructField("DEST_COUNTRY_NAME", StringType(), True), # Column_name, Datatype, Nullable
            StructField("ORIGIN_COUNTRY_NAME", StringType(), True),
            StructField("count", IntegerType(), True)
            
])


flight_df_2 = spark.read.format("csv")\
          .option("header", "true")\
          .option("SkipRows", 1)\ 
          .option("inferschema", "false")\
          .schema(my_schema_2)\
          .option('mode','FAILFAST')\
          .load("/Volumes/workspace/default/data_files/2015-summary.csv")


  File <command-7788246212618208>, line 13
    .option("SkipRows", 1)\
                            
^
SyntaxError: unexpected character after line continuation character


In [0]:
flight_df_2.show(5)

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-5024956753199229>, line 1
----> 1 flight_df_2.show(5)

NameError: name 'flight_df_2' is not defined

In [0]:
from pyspark.sql.types import StructType,StructField,StringType,IntegerType
my_schema_3 = StructType([
            StructField("DEST_COUNTRY_NAME", StringType(), True), # Column_name, Datatype, Nullable
            StructField("ORIGIN_COUNTRY_NAME", StringType(), True),
            StructField("count", IntegerType(), True)
            
])


flight_df_3 = (spark.read.format("csv")
               .option("header", "true") 
               .schema(my_schema_3) # Spark will map CSV columns to your schema by position
               .option("mode", "PERMISSIVE")
               .load("/Volumes/workspace/default/data_files/2015-summary.csv"))

flight_df_3.show(5)


+-----------------+-------------------+-----+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|
+-----------------+-------------------+-----+
|    United States|            Romania|   15|
|    United States|            Croatia|    1|
|    United States|            Ireland|  344|
|            Egypt|      United States|   15|
|    United States|              India|   62|
+-----------------+-------------------+-----+
only showing top 5 rows


###### Standard PySpark does not use SkipRows. If we have header set to true, Spark already skips the first row. By adding an extra (and likely invalid) option, the parser can get misaligned.

###### FAILFAST,PERMISSIVE and DROPMALFORMATED

In [0]:
employee_df_FAILFAST = (spark.read.format("csv")\
               .option("header", "true")\
               .option("inferschema", "false")\
               .option("mode", "FAILFAST")\
               .load("/Volumes/workspace/default/data_files/Employee_data.csv"))

employee_df_FAILFAST.show(5)

---------------------------------------------------------------------------
SparkException                            Traceback (most recent call last)
File <command-5024956753199235>, line 7
      1 employee_df_FAILFAST = (spark.read.format("csv")\
      2                .option("header", "true")\
      3                .option("inferschema", "false")\
      4                .option("mode", "FAILFAST")\
      5                .load("/Volumes/workspace/default/data_files/Employee_data.csv"))
----> 7 employee_df_FAILFAST.show(5)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:1156, in DataFrame.show(self, n, truncate, vertical)
   1155 def show(self, n: int = 20, truncate: Union[bool, int] = True, vertical: bool = False) -> None:
-> 1156     print(self._show_string(n, truncate, vertical))

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:909, in DataFrame._show_string(self, n, truncate, vertical)
    892     exc

The dataset has errors (Corrupted Records),If we use FAILFAST it gives error while loading the file gives [FAILED_READ_FILE.NO_HINT] Error while reading file dbfs:/Volumes/workspace/default/data_files/Employee_data.csv.  SQLSTATE: KD001

In [0]:
employee_df_PERMISSIVE = (spark.read.format("csv")\
               .option("header", "true")\
               .option("inferschema", "false")\
               .option("mode", "PERMISSIVE")\
               .load("/Volumes/workspace/default/data_files/Employee_data.csv"))

employee_df_PERMISSIVE.show(5)

+---+--------+---+------+------------+--------+
| id|    name|age|salary|     address| nominee|
+---+--------+---+------+------------+--------+
|  1|  Manish| 26| 75000|       bihar|nominee1|
|  2|  Nikita| 23|100000|uttarpradesh|nominee2|
|  3|  Pritam| 22|150000|   Bangalore|   India|
|  4|Prantosh| 17|200000|     Kolkata|   India|
|  5|  Vikash| 31|300000|        NULL|nominee5|
+---+--------+---+------+------------+--------+



In [0]:
employee_df_dropmalformed = (spark.read.format("csv")\
               .option("header", "true")\
               .option("inferschema", "false")\
               .option("mode", "dropmalformed")\
               .load("/Volumes/workspace/default/data_files/Employee_data.csv"))

employee_df_dropmalformed.show(5)

+---+------+---+------+------------+--------+
| id|  name|age|salary|     address| nominee|
+---+------+---+------+------------+--------+
|  1|Manish| 26| 75000|       bihar|nominee1|
|  2|Nikita| 23|100000|uttarpradesh|nominee2|
|  5|Vikash| 31|300000|        NULL|nominee5|
+---+------+---+------+------------+--------+



Failfast -- will fail to load the file,If the file has any corrupted records.

PERMISSIVE -- assign null values to the records that are corrupted and load the data, No data loss.

Dropmalformed ---- drops the corrupted records and load the file.

####### Handel corrupted records

In [0]:
# Printing corrupted records.

from pyspark.sql.types import StructType,StructField,StringType,IntegerType

corruption_detect_employee_schema = StructType([
            StructField("ID", IntegerType(), True), 
            StructField("NAME", StringType(), True), 
            StructField("AGE", IntegerType(), True), 
            StructField("SALARY", IntegerType(), True),
            StructField("ADDRESS", StringType(), True),
            StructField("NOMINEE", StringType(), True),                   
            StructField("_COURRAPT_RECORD", StringType(), True)])

employee_corruption_detect_schema = (spark.read.format("csv")\
               .option("header", "true")\
               .option("inferschema", "true")\
               .option("mode", "PERMISSIVE")\
               .schema(corruption_detect_employee_schema)\
               .load("/Volumes/workspace/default/data_files/Employee_data.csv"))

employee_corruption_detect_schema.show()

+---+--------+---+------+------------+--------+----------------+
| ID|    NAME|AGE|SALARY|     ADDRESS| NOMINEE|_COURRAPT_RECORD|
+---+--------+---+------+------------+--------+----------------+
|  1|  Manish| 26| 75000|       bihar|nominee1|            NULL|
|  2|  Nikita| 23|100000|uttarpradesh|nominee2|            NULL|
|  3|  Pritam| 22|150000|   Bangalore|   India|        nominee3|
|  4|Prantosh| 17|200000|     Kolkata|   India|        nominee4|
|  5|  Vikash| 31|300000|        NULL|nominee5|            NULL|
+---+--------+---+------+------------+--------+----------------+



In [0]:
employee_corruption_detect_schema.show(truncate=False) # default truncate=True
#if the value is to large it shows part of the value then ...., to show the full value we use truncate=False

+---+--------+---+------+------------+--------+----------------+
|ID |NAME    |AGE|SALARY|ADDRESS     |NOMINEE |_COURRAPT_RECORD|
+---+--------+---+------+------------+--------+----------------+
|1  |Manish  |26 |75000 |bihar       |nominee1|NULL            |
|2  |Nikita  |23 |100000|uttarpradesh|nominee2|NULL            |
|3  |Pritam  |22 |150000|Bangalore   |India   |nominee3        |
|4  |Prantosh|17 |200000|Kolkata     |India   |nominee4        |
|5  |Vikash  |31 |300000|NULL        |nominee5|NULL            |
+---+--------+---+------+------------+--------+----------------+



###### Store the Corrupted records for large files.(Detecting the Corrupted file name.)

fs --is the file system, (dbutils.fs)

ls --- files list 

then the path

In [0]:
%fs
ls /Volumes/workspace/default/data_files/

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/data_files/2015-summary.csv,2015-summary.csv,7080,1775730449000
dbfs:/Volumes/workspace/default/data_files/Employee_data.csv,Employee_data.csv,230,1776145590000
dbfs:/Volumes/workspace/default/data_files/Multi_line_correct.json,Multi_line_correct.json,310,1776165645000
dbfs:/Volumes/workspace/default/data_files/Multi_line_incorrect.json,Multi_line_incorrect.json,304,1776165645000
dbfs:/Volumes/workspace/default/data_files/corrupted_data/,corrupted_data/,0,1779868633703
dbfs:/Volumes/workspace/default/data_files/corrupted_json.json,corrupted_json.json,218,1776165414000
dbfs:/Volumes/workspace/default/data_files/file5.json,file5.json,669503,1776229378000
dbfs:/Volumes/workspace/default/data_files/line_delimited_json.json,line_delimited_json.json,219,1776163722000
dbfs:/Volumes/workspace/default/data_files/part-r-00000-1a9822ba-b8fb-4d8e-844a-ea30d0801b9e.gz.parquet,part-r-00000-1a9822ba-b8fb-4d8e-844a-ea30d0801b9e.gz.parquet,3921,1776328873000
dbfs:/Volumes/workspace/default/data_files/single_file_json.json,single_file_json.json,232,1776164511000


In [0]:
%fs
ls /Volumes/workspace/default/data_files

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/data_files/2015-summary.csv,2015-summary.csv,7080,1775730449000
dbfs:/Volumes/workspace/default/data_files/Employee_data.csv,Employee_data.csv,230,1776145590000
dbfs:/Volumes/workspace/default/data_files/Multi_line_correct.json,Multi_line_correct.json,310,1776165645000
dbfs:/Volumes/workspace/default/data_files/Multi_line_incorrect.json,Multi_line_incorrect.json,304,1776165645000
dbfs:/Volumes/workspace/default/data_files/corrupted_data/,corrupted_data/,0,1779868654274
dbfs:/Volumes/workspace/default/data_files/corrupted_json.json,corrupted_json.json,218,1776165414000
dbfs:/Volumes/workspace/default/data_files/file5.json,file5.json,669503,1776229378000
dbfs:/Volumes/workspace/default/data_files/line_delimited_json.json,line_delimited_json.json,219,1776163722000
dbfs:/Volumes/workspace/default/data_files/part-r-00000-1a9822ba-b8fb-4d8e-844a-ea30d0801b9e.gz.parquet,part-r-00000-1a9822ba-b8fb-4d8e-844a-ea30d0801b9e.gz.parquet,3921,1776328873000
dbfs:/Volumes/workspace/default/data_files/single_file_json.json,single_file_json.json,232,1776164511000


In [0]:
from pyspark.sql.types import StructType,StructField,StringType,IntegerType

corrupted_detect_employee_schema = StructType([
            StructField("ID", IntegerType(), True), 
            StructField("NAME", StringType(), True), 
            StructField("AGE", IntegerType(), True), 
            StructField("SALARY", IntegerType(), True),
            StructField("ADDRESS", StringType(), True),
            StructField("NOMINEE", StringType(), True),                   
            StructField("_COURRAPT_RECORD", StringType(), True)])

employee_corruption_detect_schema_store = (spark.read.format("csv")\
               .option("header", "true")\
               .option("inferschema", "true")\
               #.option("mode", "PERMISSIVE")\ -- dont use this load type if you use badRedecordsPath
               .option("badRecordsPath","/Volumes/workspace/default/data_files/corrupted_data")  # the file will saved as a json format   
               .schema(corrupted_detect_employee_schema)\
               .load("/Volumes/workspace/default/data_files/Employee_data.csv"))

employee_corruption_detect_schema_store.show(truncate=False)

+---+--------+---+------+---------+-------+----------------+
|ID |NAME    |AGE|SALARY|ADDRESS  |NOMINEE|_COURRAPT_RECORD|
+---+--------+---+------+---------+-------+----------------+
|3  |Pritam  |22 |150000|Bangalore|India  |nominee3        |
|4  |Prantosh|17 |200000|Kolkata  |India  |nominee4        |
+---+--------+---+------+---------+-------+----------------+



In [0]:
%fs
ls /Volumes/workspace/default/data_files

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/data_files/2015-summary.csv,2015-summary.csv,7080,1775730449000
dbfs:/Volumes/workspace/default/data_files/Employee_data.csv,Employee_data.csv,230,1776145590000
dbfs:/Volumes/workspace/default/data_files/Multi_line_correct.json,Multi_line_correct.json,310,1776165645000
dbfs:/Volumes/workspace/default/data_files/Multi_line_incorrect.json,Multi_line_incorrect.json,304,1776165645000
dbfs:/Volumes/workspace/default/data_files/corrupted_data/,corrupted_data/,0,1779868715829
dbfs:/Volumes/workspace/default/data_files/corrupted_json.json,corrupted_json.json,218,1776165414000
dbfs:/Volumes/workspace/default/data_files/file5.json,file5.json,669503,1776229378000
dbfs:/Volumes/workspace/default/data_files/line_delimited_json.json,line_delimited_json.json,219,1776163722000
dbfs:/Volumes/workspace/default/data_files/part-r-00000-1a9822ba-b8fb-4d8e-844a-ea30d0801b9e.gz.parquet,part-r-00000-1a9822ba-b8fb-4d8e-844a-ea30d0801b9e.gz.parquet,3921,1776328873000
dbfs:/Volumes/workspace/default/data_files/single_file_json.json,single_file_json.json,232,1776164511000


In [0]:
%fs
ls /Volumes/workspace/default/data_files/corrupted_data

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/data_files/corrupted_data/20260414T065443/,20260414T065443/,0,1779868720335


In [0]:
%fs
ls /Volumes/workspace/default/data_files/corrupted_data/20260414T065443

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/data_files/corrupted_data/20260414T065443/bad_records/,bad_records/,0,1779868730541


Exact File path

In [0]:
%fs
ls /Volumes/workspace/default/data_files/corrupted_data/20260414T065443/bad_records/

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/data_files/corrupted_data/20260414T065443/bad_records/part-00000-6bfd77aa-175d-45ad-b459-d0ecde9a86c0,part-00000-6bfd77aa-175d-45ad-b459-d0ecde9a86c0,568,1776149684000


In [0]:
corrupted_data_df = spark.read.format("json")\
               .option("header", "true")\
               .option("inferschema", "true")\
               .load("/Volumes/workspace/default/data_files/corrupted_data/20260414T065443/bad_records/")


corrupted_data_df.show(truncate=False)

+------------------------------------------------------------+------------------------------------------------------------+----------------------------------------+
|path                                                        |reason                                                      |record                                  |
+------------------------------------------------------------+------------------------------------------------------------+----------------------------------------+
|dbfs:/Volumes/workspace/default/data_files/Employee_data.csv|org.apache.spark.sql.catalyst.util.LazyBadRecordCauseWrapper|1,Manish,26,75000,bihar,nominee1        |
|dbfs:/Volumes/workspace/default/data_files/Employee_data.csv|org.apache.spark.sql.catalyst.util.LazyBadRecordCauseWrapper|2,Nikita,23,100000,uttarpradesh,nominee2|
|dbfs:/Volumes/workspace/default/data_files/Employee_data.csv|org.apache.spark.sql.catalyst.util.LazyBadRecordCauseWrapper|5,Vikash,31,300000,,nominee5            |
+---------

Remove the Corrupted folder

%fs

rm -r /Volumes/workspace/default/data_files/corrupted_data/

##### Transformations and Actions with DAG (DIRECTED ACYCLIC GRAPH)

In [0]:
# This is a spark application
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

# 1. Load the data (Transformation - Lazy) (format and inferSchema are two actions)
flight_data = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/Volumes/workspace/default/data_files/2015-summary.csv")

# 2. Repartition the data (Wide Transformation)
# We assign it to a variable so we can use the optimized version
flight_data_repartition = flight_data.repartition(3)

# 3. Filter for United States destinations (Narrow Transformation)
us_flight_data = flight_data_repartition.filter("DEST_COUNTRY_NAME == 'United States'") # in one string DEST_COUNTRY_NAME == 'United States'

# 4. Filter for India or Singapore origins (Narrow Transformation)
us_india_data = us_flight_data.filter(
    (col("ORIGIN_COUNTRY_NAME") == 'India') | 
    (col("ORIGIN_COUNTRY_NAME") == 'Singapore')
)

# 5. Group by and Sum (Wide Transformation)
total_flight_ind_sing = us_india_data.groupby("DEST_COUNTRY_NAME").sum("count")

# 6. Show the results (Action - Triggers the Job)
total_flight_ind_sing.show()

+-----------------+----------+
|DEST_COUNTRY_NAME|sum(count)|
+-----------------+----------+
|    United States|        63|
+-----------------+----------+



##### Working with JSON Data

In [0]:
line_delimited_json = spark.read.format("json") \
    .option("inferSchema", "true") \
    .option("mode", "PERMISSIVE") \
    .load("/Volumes/workspace/default/data_files/line_delimited_json.json")

line_delimited_json.show()

+----+--------+------+
| age|    name|salary|
+----+--------+------+
|  20|  Manish| 20000|
|NULL|    NULL|  NULL|
|  25|  Nikita| 21000|
|  16|  Pritam| 22000|
|  35|Prantosh| 25000|
|  67|  Vikash| 40000|
+----+--------+------+



In [0]:
single_file_json = spark.read.format("json") \
    .option("inferSchema", "true") \
    .option("mode", "dropmalaformed") \
.load("/Volumes/workspace/default/data_files/single_file_json.json")


single_file_json.show()

+----+------+--------+------+
| age|gender|    name|salary|
+----+------+--------+------+
|  20|  NULL|  Manish| 20000|
|NULL|  NULL|    NULL|  NULL|
|  25|  NULL|  Nikita| 21000|
|  16|  NULL|  Pritam| 22000|
|  35|  NULL|Prantosh| 25000|
|  67|     M|  Vikash| 40000|
+----+------+--------+------+



In [0]:
Multi_line_incorrect = spark.read.format("json") \
    .option("inferSchema", "true") \
    .option("mode", "PERMISSIVE") \
    .option("multiLine", "false") \
.load("/Volumes/workspace/default/data_files/Multi_line_incorrect.json")

Multi_line_incorrect.show()

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-4998873548944745>, line 7
      1 Multi_line_incorrect = spark.read.format("json") \
      2     .option("inferSchema", "true") \
      3     .option("mode", "PERMISSIVE") \
      4     .option("multiLine", "false") \
      5 .load("/Volumes/workspace/default/data_files/Multi_line_incorrect.json")
----> 7 Multi_line_incorrect.show()

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:1156, in DataFrame.show(self, n, truncate, vertical)
   1155 def show(self, n: int = 20, truncate: Union[bool, int] = True, vertical: bool = False) -> None:
-> 1156     print(self._show_string(n, truncate, vertical))

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:909, in DataFrame._show_string(self, n, truncate, vertical)
    892     except ValueError:
    893         r

In [0]:
Multi_line_incorrect = spark.read.format("json") \
    .option("inferSchema", "true") \
    .option("mode", "PERMISSIVE") \
    .option("multiLine", "true") \
.load("/Volumes/workspace/default/data_files/Multi_line_incorrect.json")

Multi_line_incorrect.show()

+----+------+------+
| age|  name|salary|
+----+------+------+
|  20|Manish| 20000|
|NULL|  NULL|  NULL|
+----+------+------+



In [0]:
Multi_line_correct = spark.read.format("json") \
    .option("inferSchema", "true") \
    .option("mode", "PERMISSIVE") \
    .option("multiLine", "true") \
.load("/Volumes/workspace/default/data_files/Multi_line_correct.json")

Multi_line_correct.show()

+---+--------+------+
|age|    name|salary|
+---+--------+------+
| 20|  Manish| 20000|
| 25|  Nikita| 21000|
| 16|  Pritam| 22000|
| 35|Prantosh| 25000|
| 67|  Vikash| 40000|
+---+--------+------+



In [0]:
corrupted_json = spark.read.format("json") \
    .option("inferSchema", "true") \
    .option("mode", "PERMISSIVE") \
.load("/Volumes/workspace/default/data_files/corrupted_json.json")

corrupted_json.show(truncate=False)

+----------------------------------------+----+--------+------+
|_corrupt_record                         |age |name    |salary|
+----------------------------------------+----+--------+------+
|NULL                                    |20  |Manish  |20000 |
|NULL                                    |25  |Nikita  |21000 |
|NULL                                    |16  |Pritam  |22000 |
|NULL                                    |35  |Prantosh|25000 |
|{"name":"Vikash","age":67,"salary":40000|NULL|NULL    |NULL  |
+----------------------------------------+----+--------+------+



In [0]:
Nasted_json = spark.read.format("json") \
         .option("inferSchema", "true") \
    .option("mode", "PERMISSIVE") \
    .option("multiLine", "true") \
.load("/Volumes/workspace/default/data_files/file5.json")

Nasted_json.show()

+----+-------+--------------------+-------------+-------------+-------------+------+
|code|message|         restaurants|results_found|results_shown|results_start|status|
+----+-------+--------------------+-------------+-------------+-------------+------+
|NULL|   NULL|                  []|            0|            0|            1|  NULL|
|NULL|   NULL|[{{{17066603}, b9...|         6835|           20|            1|  NULL|
|NULL|   NULL|                  []|            0|            0|            1|  NULL|
|NULL|   NULL|                  []|            0|            0|            1|  NULL|
|NULL|   NULL|[{{{17093124}, b9...|         8680|           20|            1|  NULL|
|NULL|   NULL|                  []|            0|            0|            1|  NULL|
|NULL|   NULL|                  []|            0|            0|            1|  NULL|
|NULL|   NULL|[{{{17580142}, b9...|          943|           20|            1|  NULL|
|NULL|   NULL|                  []|            0|            0|  

In [0]:
Nasted_json.printSchema()

root
 |-- code: long (nullable = true)
 |-- message: string (nullable = true)
 |-- restaurants: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- restaurant: struct (nullable = true)
 |    |    |    |-- R: struct (nullable = true)
 |    |    |    |    |-- res_id: long (nullable = true)
 |    |    |    |-- apikey: string (nullable = true)
 |    |    |    |-- average_cost_for_two: long (nullable = true)
 |    |    |    |-- cuisines: string (nullable = true)
 |    |    |    |-- currency: string (nullable = true)
 |    |    |    |-- deeplink: string (nullable = true)
 |    |    |    |-- establishment_types: array (nullable = true)
 |    |    |    |    |-- element: string (containsNull = true)
 |    |    |    |-- events_url: string (nullable = true)
 |    |    |    |-- featured_image: string (nullable = true)
 |    |    |    |-- has_online_delivery: long (nullable = true)
 |    |    |    |-- has_table_booking: long (nullable = true)
 |    |    |    |-- i

##### Spark API Engine (Stucture API Execuation)

Analysis eception error (catalog optimizer)

In [0]:
# Analysis eception error
df = spark.read.format("json") \
         .option("inferSchema", "true") \
    .option("mode", "PERMISSIVE") \
    .option("multiLine", "true") \
.load("/Volumes/workspace/default/data_files/file5.json")

df.select(name1).show()

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-4998873548944743>, line 8
      1 # Analysis eception error
      2 df = spark.read.format("json") \
      3          .option("inferSchema", "true") \
      4     .option("mode", "PERMISSIVE") \
      5     .option("multiLine", "true") \
      6 .load("/Volumes/workspace/default/data_files/file5.json")
----> 8 df.select(name1).show()

NameError: name 'name1' is not defined

###### RDD (Resilient Distributed Dataset)

###### Parquet file format.(The data in it in binary file format and column based file format. OLAP(Online Analytical Processing))


In [0]:
parquet_df = spark.read.parquet\
("/Volumes/workspace/default/data_files/part-r-00000-1a9822ba-b8fb-4d8e-844a-ea30d0801b9e.gz.parquet")

parquet_df.show() # it has all meta data so we do not need to give any othe info.

+--------------------+-------------------+-----+
|   DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|
+--------------------+-------------------+-----+
|       United States|            Romania|    1|
|       United States|            Ireland|  264|
|       United States|              India|   69|
|               Egypt|      United States|   24|
|   Equatorial Guinea|      United States|    1|
|       United States|          Singapore|   25|
|       United States|            Grenada|   54|
|          Costa Rica|      United States|  477|
|             Senegal|      United States|   29|
|       United States|   Marshall Islands|   44|
|              Guyana|      United States|   17|
|       United States|       Sint Maarten|   53|
|               Malta|      United States|    1|
|             Bolivia|      United States|   46|
|            Anguilla|      United States|   21|
|Turks and Caicos ...|      United States|  136|
|       United States|        Afghanistan|    2|
|Saint Vincent and..

In [0]:
!pip install pyarrow

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import pyarrow as pa
import pyarrow.parquet as pq
parquet_file = pq.ParquetFile("/Volumes/workspace/default/data_files/part-r-00000-1a9822ba-b8fb-4d8e-844a-ea30d0801b9e.gz.parquet")
print(parquet_file.schema)
metadata =parquet_file.metadata
print(metadata)
print(parquet_file.read_row_group(0))
print(parquet_file.read_row_group(0).column(0))


required group field_id=-1 spark_schema {
  optional binary field_id=-1 DEST_COUNTRY_NAME (String);
  optional binary field_id=-1 ORIGIN_COUNTRY_NAME (String);
  optional int64 field_id=-1 count;
}

  created_by: parquet-mr (build 32c46643845ea8a705c35d4ec8fc654cc8ff816d)
  num_columns: 3
  num_rows: 255
  num_row_groups: 1
  format_version: 1.0
  serialized_size: 658
pyarrow.Table
DEST_COUNTRY_NAME: string
ORIGIN_COUNTRY_NAME: string
count: int64
----
DEST_COUNTRY_NAME: [["United States","United States","United States","Egypt","Equatorial Guinea",...,"United States","United States","United States","Bonaire, Sint Eustatius, and Saba","Greece"]]
ORIGIN_COUNTRY_NAME: [["Romania","Ireland","India","United States","United States",...,"French Guiana","Haiti","Uganda","United States","United States"]]
count: [[1,264,69,24,1,...,1,226,1,16,50]]
[
  [
    "United States",
    "United States",
    "United States",
    "Egypt",
    "Equatorial Guinea",
    ...
    "United States",
    "United St

In [0]:
for group_idx in range(metadata.num_row_groups):
    row_group = metadata.row_group(group_idx)
    print(f"\n--- Row Group {group_idx} (Rows: {row_group.num_rows}) ---")
    
    for col_idx in range(row_group.num_columns):
        col_chunk = row_group.column(col_idx)
        stats = col_chunk.statistics
        
        print(f"Column: {col_chunk.path_in_schema}")
        if stats and stats.has_min_max:
            print(f"  Min: {stats.min}")
            print(f"  Max: {stats.max}")
            print(f"  Null Count: {stats.null_count}")
        else:
            print("  No statistics available for this column.")


--- Row Group 0 (Rows: 255) ---
Column: DEST_COUNTRY_NAME
  Min: Afghanistan
  Max: Vietnam
  Null Count: 0
Column: ORIGIN_COUNTRY_NAME
  Min: Afghanistan
  Max: Vietnam
  Null Count: 0
Column: count
  Min: 1
  Max: 348113
  Null Count: 0


#### Dataframe Writer

df.write.format('file_format')\
    .option('hader','true')\
    .option('mode','overwrite')\
    .partitionBy('partition_column')\
    .bucketBy()\
    .save('/Volumes/workspace/default/data_files/')

Modes in dataframe api:
1. Append
2. overwite
3. error if exists
4. ignore

In [0]:
df = (
    spark.read
         .format("csv")
         .option("header", "true")
         .option("inferSchema", "true")
         .load("/Volumes/workspace/default/data_files/employee_data.csv")
)

display(df)

id,name,age,salary,address,gender
1,Manish,26,75000,INDIA,m
2,Nikita,23,100000,USA,f
3,Pritam,22,150000,INDIA,m
4,Prantosh,17,200000,JAPAN,m
5,Vikash,31,300000,USA,m
6,Rahul,55,300000,INDIA,m
7,Raju,67,540000,USA,m
8,Praveen,28,70000,JAPAN,m
9,Dev,32,150000,JAPAN,m
10,Sherin,16,25000,RUSSIA,f


In [0]:
df.show()

+---+--------+---+------+-------+------+
| id|    name|age|salary|address|gender|
+---+--------+---+------+-------+------+
|  1|  Manish| 26| 75000|  INDIA|     m|
|  2|  Nikita| 23|100000|    USA|     f|
|  3|  Pritam| 22|150000|  INDIA|     m|
|  4|Prantosh| 17|200000|  JAPAN|     m|
|  5|  Vikash| 31|300000|    USA|     m|
|  6|   Rahul| 55|300000|  INDIA|     m|
|  7|    Raju| 67|540000|    USA|     m|
|  8| Praveen| 28| 70000|  JAPAN|     m|
|  9|     Dev| 32|150000|  JAPAN|     m|
| 10|  Sherin| 16| 25000| RUSSIA|     f|
| 11|    Ragu| 12| 35000|  INDIA|     f|
| 12|   Sweta| 43|200000|  INDIA|     f|
| 13| Raushan| 48|650000|    USA|     m|
| 14|  Mukesh| 36| 95000| RUSSIA|     m|
| 15| Prakash| 52|750000|  INDIA|     m|
+---+--------+---+------+-------+------+



In [0]:
# by default its takes the parquet file format
df.write.format('csv')\
.option('header',"true")\
.mode("overwrite")\
.save('/Volumes/workspace/default/data_files/write/')

In [0]:
# checking the file is created or not

dbutils.fs.ls("/Volumes/workspace/default/data_files/write/")

[FileInfo(path='dbfs:/Volumes/workspace/default/data_files/write/_SUCCESS', name='_SUCCESS', size=0, modificationTime=1781266753000),
 FileInfo(path='dbfs:/Volumes/workspace/default/data_files/write/_committed_4580664960891198859', name='_committed_4580664960891198859', size=212, modificationTime=1781266743000),
 FileInfo(path='dbfs:/Volumes/workspace/default/data_files/write/_committed_7341486344419585721', name='_committed_7341486344419585721', size=201, modificationTime=1781266753000),
 FileInfo(path='dbfs:/Volumes/workspace/default/data_files/write/_committed_9079457729522438532', name='_committed_9079457729522438532', size=113, modificationTime=1781264190000),
 FileInfo(path='dbfs:/Volumes/workspace/default/data_files/write/_committed_vacuum6750171427665840812', name='_committed_vacuum6750171427665840812', size=96, modificationTime=1781266743000),
 FileInfo(path='dbfs:/Volumes/workspace/default/data_files/write/_started_4580664960891198859', name='_started_4580664960891198859', si

In [0]:
# by default its takes the parquet file format
df.repartition(3).write\
.format('csv')\
.option('header',"true")\
.mode("overwrite")\
.save('/Volumes/workspace/default/data_files/csv_repartition/')

In [0]:
# checking the file is created or not

dbutils.fs.ls("/Volumes/workspace/default/data_files/csv_repartition/")

[FileInfo(path='dbfs:/Volumes/workspace/default/data_files/csv_repartition/_SUCCESS', name='_SUCCESS', size=0, modificationTime=1781266774000),
 FileInfo(path='dbfs:/Volumes/workspace/default/data_files/csv_repartition/_committed_2853154682086239162', name='_committed_2853154682086239162', size=291, modificationTime=1781264332000),
 FileInfo(path='dbfs:/Volumes/workspace/default/data_files/csv_repartition/_committed_5794196649634644301', name='_committed_5794196649634644301', size=568, modificationTime=1781266774000),
 FileInfo(path='dbfs:/Volumes/workspace/default/data_files/csv_repartition/_committed_vacuum9098429383699025906', name='_committed_vacuum9098429383699025906', size=96, modificationTime=1781266774000),
 FileInfo(path='dbfs:/Volumes/workspace/default/data_files/csv_repartition/_started_5794196649634644301', name='_started_5794196649634644301', size=0, modificationTime=1781266774000),
 FileInfo(path='dbfs:/Volumes/workspace/default/data_files/csv_repartition/part-00000-tid-5

In [0]:
# by default its takes the parquet file format
df.repartition(3).write\
.format('csv')\
.option('header',"true")\
.partitionBy("Address")\
.mode("overwrite")\
.save('/Volumes/workspace/default/data_files/partition_by_addresss/')

In [0]:
dbutils.fs.ls('/Volumes/workspace/default/data_files/partition_by_addresss/')

[FileInfo(path='dbfs:/Volumes/workspace/default/data_files/partition_by_addresss/Address=INDIA/', name='Address=INDIA/', size=0, modificationTime=1781267563749),
 FileInfo(path='dbfs:/Volumes/workspace/default/data_files/partition_by_addresss/Address=JAPAN/', name='Address=JAPAN/', size=0, modificationTime=1781267563749),
 FileInfo(path='dbfs:/Volumes/workspace/default/data_files/partition_by_addresss/Address=RUSSIA/', name='Address=RUSSIA/', size=0, modificationTime=1781267563749),
 FileInfo(path='dbfs:/Volumes/workspace/default/data_files/partition_by_addresss/Address=USA/', name='Address=USA/', size=0, modificationTime=1781267563749),
 FileInfo(path='dbfs:/Volumes/workspace/default/data_files/partition_by_addresss/_SUCCESS', name='_SUCCESS', size=0, modificationTime=1781266778000),
 FileInfo(path='dbfs:/Volumes/workspace/default/data_files/partition_by_addresss/_committed_1769888397648197977', name='_committed_1769888397648197977', size=35, modificationTime=1781266778000)]

In [0]:
# .partitionBy("Address","Gender")\ -- first we have a folder with name Address inside that we have Gender
df.repartition(3).write\
.format('csv')\
.option('header',"true")\
.partitionBy("Address","Gender")\
.mode("overwrite")\
.save('/Volumes/workspace/default/data_files/partition_by_addresss_gender/')

In [0]:
dbutils.fs.ls('/Volumes/workspace/default/data_files/partition_by_addresss_gender/')

[FileInfo(path='dbfs:/Volumes/workspace/default/data_files/partition_by_addresss_gender/Address=INDIA/', name='Address=INDIA/', size=0, modificationTime=1781270889688),
 FileInfo(path='dbfs:/Volumes/workspace/default/data_files/partition_by_addresss_gender/Address=JAPAN/', name='Address=JAPAN/', size=0, modificationTime=1781270889688),
 FileInfo(path='dbfs:/Volumes/workspace/default/data_files/partition_by_addresss_gender/Address=RUSSIA/', name='Address=RUSSIA/', size=0, modificationTime=1781270889688),
 FileInfo(path='dbfs:/Volumes/workspace/default/data_files/partition_by_addresss_gender/Address=USA/', name='Address=USA/', size=0, modificationTime=1781270889688),
 FileInfo(path='dbfs:/Volumes/workspace/default/data_files/partition_by_addresss_gender/_SUCCESS', name='_SUCCESS', size=0, modificationTime=1781270888000)]

In [0]:
dbutils.fs.ls('/Volumes/workspace/default/data_files/partition_by_addresss_gender/Address=INDIA/')

[FileInfo(path='dbfs:/Volumes/workspace/default/data_files/partition_by_addresss_gender/Address=INDIA/Gender=f/', name='Gender=f/', size=0, modificationTime=1781270969101),
 FileInfo(path='dbfs:/Volumes/workspace/default/data_files/partition_by_addresss_gender/Address=INDIA/Gender=m/', name='Gender=m/', size=0, modificationTime=1781270969101)]

In [0]:
# If the data has no cardinality so at that case we have to store as bucketed

df.write\
.format('csv')\
.option('header',"true")\
.mode("overwrite")\
.bucketBy(3,"id")\
.save('/Volumes/workspace/default/data_files/bucketed/BucketBy_Id')

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-4785866282930909>, line 8
      1 # If the data has no cardinality so at that case we have to store as bucketed table
      3 df.write\
      4 .format('csv')\
      5 .option('header',"true")\
      6 .mode("overwrite")\
      7 .bucketBy(3,"id")\
----> 8 .save('/Volumes/workspace/default/data_files/bucketed/BucketBy_Id')

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/readwriter.py:703, in DataFrameWriter.save(self, path, format, mode, partitionBy, **options)
    701     self.format(format)
    702 self._write.path = path
--> 703 _, _, ei = self._spark.client.execute_command(
    704     self._write.command(self._spark.client), self._write.observations
    705 )
    706 self._callback(ei)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/client/core.py:1538, in SparkConne

In [0]:
# If the data has no cardinality so at that case we have to store as bucketed table

df.write\
.format('csv')\
.option('header',"true")\
.mode("overwrite")\
.bucketBy(3,"id")\
.option('path',"/Volumes/workspace/default/data_files/bucketed/")\
.saveAsTable('Bucket_By_Id')

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-4785866282930911>, line 9
      1 # If the data has no cardinality so at that case we have to store as bucketed table
      3 df.write\
      4 .format('csv')\
      5 .option('header',"true")\
      6 .mode("overwrite")\
      7 .bucketBy(3,"id")\
      8 .option('path',"/Volumes/workspace/default/data_files/bucketed/")\
----> 9 .saveAsTable('Bucket_By_Id')

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/readwriter.py:737, in DataFrameWriter.saveAsTable(self, name, format, mode, partitionBy, **options)
    735 self._write.table_name = name
    736 self._write.table_save_method = "save_as_table"
--> 737 _, _, ei = self._spark.client.execute_command(
    738     self._write.command(self._spark.client), self._write.observations
    739 )
    740 self._callback(ei)

File /databricks/python/lib/python

In [0]:
employee_df = spark.read \
    .format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/Volumes/workspace/default/data_files/employee_data.csv")

In [0]:
df.write \
    .format("parquet") \
    .mode("overwrite") \
    .save("/Volumes/workspace/default/data_files/employee_parquet")

In [0]:
df.write \
  .mode("overwrite") \
  .bucketBy(3, "id") \
  .saveAsTable("bucket_by_id")

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-8710628263027516>, line 4
      1 df.write \
      2   .mode("overwrite") \
      3   .bucketBy(3, "id") \
----> 4   .saveAsTable("bucket_by_id")

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/readwriter.py:737, in DataFrameWriter.saveAsTable(self, name, format, mode, partitionBy, **options)
    735 self._write.table_name = name
    736 self._write.table_save_method = "save_as_table"
--> 737 _, _, ei = self._spark.client.execute_command(
    738     self._write.command(self._spark.client), self._write.observations
    739 )
    740 self._callback(ei)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/client/core.py:1538, in SparkConnectClient.execute_command(self, command, observations, extra_request_metadata)
   1536     req.user_context.user_id = self._user_id
   1537 sel

create dataframe in pyspark:

process of data engineering : 
1. read
2. transformation
3. write
4. bi/model

in transformation we have two process for transformations in high level :

a. dataframe api

b. spark sql

create a dataframe in spark

In [0]:
my_data = [ (1  , 1),
 (2  , 1),
 (3 ,  1),
 (4  , 2),
 (5  , 1),
 (6 ,  2),
 (7  , 2)]

In [0]:
my_schema =["id","num"]

In [0]:
spark.createDataFrame(data=my_data,schema=my_schema).show()

+---+---+
| id|num|
+---+---+
|  1|  1|
|  2|  1|
|  3|  1|
|  4|  2|
|  5|  1|
|  6|  2|
|  7|  2|
+---+---+



In [0]:
employee_df = spark.read \
    .format("csv") \
    .option("inferSchema", "true")\
    .option("header", "true") \
    .load("/Volumes/workspace/default/data_files/employee_data.csv")

In [0]:
employee_df.printSchema()

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- salary: integer (nullable = true)
 |-- address: string (nullable = true)
 |-- gender: string (nullable = true)



In [0]:
print(employee_df.columns)

['id', 'name', 'age', 'salary', 'address', 'gender']


create rows in dataframe

In [0]:
from pyspark.sql import Row

new_row = [Row(1546, "john_smith", 46, 48000, None, "M")]

row_df = spark.createDataFrame(new_row, schema=employee_df.schema)

new_employee_df = employee_df.union(row_df)

display(new_employee_df)

id,name,age,salary,address,gender
1,Manish,26,75000,INDIA,m
2,Nikita,23,100000,USA,f
3,Pritam,22,150000,INDIA,m
4,Prantosh,17,200000,JAPAN,m
5,Vikash,31,300000,USA,m
6,Rahul,55,300000,INDIA,m
7,Raju,67,540000,USA,m
8,Praveen,28,70000,JAPAN,m
9,Dev,32,150000,JAPAN,m
10,Sherin,16,25000,RUSSIA,f


Columns : Columns are expressions,i.e. set of transformations on one or more than one values in a record.

Multiple ways to select columns.

In [0]:
new_employee_df.select('name').show() # select only name column using string method

+----------+
|      name|
+----------+
|    Manish|
|    Nikita|
|    Pritam|
|  Prantosh|
|    Vikash|
|     Rahul|
|      Raju|
|   Praveen|
|       Dev|
|    Sherin|
|      Ragu|
|     Sweta|
|   Raushan|
|    Mukesh|
|   Prakash|
|john_smith|
+----------+



In [0]:
new_employee_df.select(col('name')).show() # select only name column using column method

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-4587885921184137>, line 1
----> 1 new_employee_df.select(col('name')).show() # select only name column using column method

NameError: name 'col' is not defined

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

new_employee_df.select(col('name')).show() # select only name column using column method

+----------+
|      name|
+----------+
|    Manish|
|    Nikita|
|    Pritam|
|  Prantosh|
|    Vikash|
|     Rahul|
|      Raju|
|   Praveen|
|       Dev|
|    Sherin|
|      Ragu|
|     Sweta|
|   Raushan|
|    Mukesh|
|   Prakash|
|john_smith|
+----------+



In [0]:
new_employee_df.select('id + 5').show() # select only name column using string method(analysis exception error comes from catalog)

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7990907642419657>, line 1
----> 1 new_employee_df.select('id + 5').show() # select only name column using column method

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:1156, in DataFrame.show(self, n, truncate, vertical)
   1155 def show(self, n: int = 20, truncate: Union[bool, int] = True, vertical: bool = False) -> None:
-> 1156     print(self._show_string(n, truncate, vertical))

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:909, in DataFrame._show_string(self, n, truncate, vertical)
    892     except ValueError:
    893         raise PySparkTypeError(
    894             errorClass="NOT_BOOL",
    895             messageParameters={
   (...)
    898             },
    899         )
    901 table, _ = DataFrame(
    902     plan.ShowString(


In [0]:
new_employee_df.select(col('id')+5).show() # select only name column using column method

+--------+
|(id + 5)|
+--------+
|       6|
|       7|
|       8|
|       9|
|      10|
|      11|
|      12|
|      13|
|      14|
|      15|
|      16|
|      17|
|      18|
|      19|
|      20|
|    1551|
+--------+



In [0]:
new_employee_df.select((col('id')+5).alias('id_plus_5')).show() # changing the column name column using column method

+---------+
|id_plus_5|
+---------+
|        6|
|        7|
|        8|
|        9|
|       10|
|       11|
|       12|
|       13|
|       14|
|       15|
|       16|
|       17|
|       18|
|       19|
|       20|
|     1551|
+---------+



selecting multiple columns.

In [0]:
# String Method

new_employee_df.select('id','name','salary').show()

+----+----------+------+
|  id|      name|salary|
+----+----------+------+
|   1|    Manish| 75000|
|   2|    Nikita|100000|
|   3|    Pritam|150000|
|   4|  Prantosh|200000|
|   5|    Vikash|300000|
|   6|     Rahul|300000|
|   7|      Raju|540000|
|   8|   Praveen| 70000|
|   9|       Dev|150000|
|  10|    Sherin| 25000|
|  11|      Ragu| 35000|
|  12|     Sweta|200000|
|  13|   Raushan|650000|
|  14|    Mukesh| 95000|
|  15|   Prakash|750000|
|1546|john_smith| 48000|
+----+----------+------+



In [0]:
# see all columns
new_employee_df.select('*').show()

+----+----------+---+------+-------+------+
|  id|      name|age|salary|address|gender|
+----+----------+---+------+-------+------+
|   1|    Manish| 26| 75000|  INDIA|     m|
|   2|    Nikita| 23|100000|    USA|     f|
|   3|    Pritam| 22|150000|  INDIA|     m|
|   4|  Prantosh| 17|200000|  JAPAN|     m|
|   5|    Vikash| 31|300000|    USA|     m|
|   6|     Rahul| 55|300000|  INDIA|     m|
|   7|      Raju| 67|540000|    USA|     m|
|   8|   Praveen| 28| 70000|  JAPAN|     m|
|   9|       Dev| 32|150000|  JAPAN|     m|
|  10|    Sherin| 16| 25000| RUSSIA|     f|
|  11|      Ragu| 12| 35000|  INDIA|     f|
|  12|     Sweta| 43|200000|  INDIA|     f|
|  13|   Raushan| 48|650000|    USA|     m|
|  14|    Mukesh| 36| 95000| RUSSIA|     m|
|  15|   Prakash| 52|750000|  INDIA|     m|
|1546|john_smith| 46| 48000|   NULL|     M|
+----+----------+---+------+-------+------+



In [0]:
# column Method

new_employee_df.select(col('id'),col('name'),col('salary')).show()

+----+----------+------+
|  id|      name|salary|
+----+----------+------+
|   1|    Manish| 75000|
|   2|    Nikita|100000|
|   3|    Pritam|150000|
|   4|  Prantosh|200000|
|   5|    Vikash|300000|
|   6|     Rahul|300000|
|   7|      Raju|540000|
|   8|   Praveen| 70000|
|   9|       Dev|150000|
|  10|    Sherin| 25000|
|  11|      Ragu| 35000|
|  12|     Sweta|200000|
|  13|   Raushan|650000|
|  14|    Mukesh| 95000|
|  15|   Prakash|750000|
|1546|john_smith| 48000|
+----+----------+------+



In [0]:
# combination Method (String, column,DataFrame,DataFrame)

new_employee_df.select('id',col('name'),new_employee_df.salary,new_employee_df["address"]).show()

+----+----------+------+-------+
|  id|      name|salary|address|
+----+----------+------+-------+
|   1|    Manish| 75000|  INDIA|
|   2|    Nikita|100000|    USA|
|   3|    Pritam|150000|  INDIA|
|   4|  Prantosh|200000|  JAPAN|
|   5|    Vikash|300000|    USA|
|   6|     Rahul|300000|  INDIA|
|   7|      Raju|540000|    USA|
|   8|   Praveen| 70000|  JAPAN|
|   9|       Dev|150000|  JAPAN|
|  10|    Sherin| 25000| RUSSIA|
|  11|      Ragu| 35000|  INDIA|
|  12|     Sweta|200000|  INDIA|
|  13|   Raushan|650000|    USA|
|  14|    Mukesh| 95000| RUSSIA|
|  15|   Prakash|750000|  INDIA|
|1546|john_smith| 48000|   NULL|
+----+----------+------+-------+



In [0]:
# column expression (accepts multiple columns)

new_employee_df.select((expr(' id + 5 ')).alias("id_plus_5")).show()

+---------+
|id_plus_5|
+---------+
|        6|
|        7|
|        8|
|        9|
|       10|
|       11|
|       12|
|       13|
|       14|
|       15|
|       16|
|       17|
|       18|
|       19|
|       20|
|     1551|
+---------+



In [0]:
# column expression (accepts multiple columns, we can add sql quaries inside this)
from pyspark.sql.functions import expr

new_employee_df.select(expr('id as emp_id'),(expr('concat(id,'_',name) as emp_name_id'))).show()

  File <command-5622917355670384>, line 4
    new_employee_df.select(expr('id as emp_id'),(expr('concat(id,'_',name) as emp_name_id'))).show()
                                                      ^
SyntaxError: invalid syntax. Perhaps you forgot a comma?


In [0]:
# column expression (accepts multiple columns, we can add sql quaries inside this)
from pyspark.sql.functions import expr

new_employee_df.select(expr('id as emp_id'),(expr("concat(id,'_',name) as emp_name_id"))).show()

+------+---------------+
|emp_id|    emp_name_id|
+------+---------------+
|     1|       1_Manish|
|     2|       2_Nikita|
|     3|       3_Pritam|
|     4|     4_Prantosh|
|     5|       5_Vikash|
|     6|        6_Rahul|
|     7|         7_Raju|
|     8|      8_Praveen|
|     9|          9_Dev|
|    10|      10_Sherin|
|    11|        11_Ragu|
|    12|       12_Sweta|
|    13|     13_Raushan|
|    14|      14_Mukesh|
|    15|     15_Prakash|
|  1546|1546_john_smith|
+------+---------------+



Spark SQL

* we need to create a temporary view first then by using that view we can perform spark sql operations

In [0]:
# step 1.
new_employee_df.createOrReplaceTempView('employee_table') # creating temp view
# step 2. Using spark sql
spark.sql("""
          select * from employee_table
          """).show()

+----+----------+---+------+-------+------+
|  id|      name|age|salary|address|gender|
+----+----------+---+------+-------+------+
|   1|    Manish| 26| 75000|  INDIA|     m|
|   2|    Nikita| 23|100000|    USA|     f|
|   3|    Pritam| 22|150000|  INDIA|     m|
|   4|  Prantosh| 17|200000|  JAPAN|     m|
|   5|    Vikash| 31|300000|    USA|     m|
|   6|     Rahul| 55|300000|  INDIA|     m|
|   7|      Raju| 67|540000|    USA|     m|
|   8|   Praveen| 28| 70000|  JAPAN|     m|
|   9|       Dev| 32|150000|  JAPAN|     m|
|  10|    Sherin| 16| 25000| RUSSIA|     f|
|  11|      Ragu| 12| 35000|  INDIA|     f|
|  12|     Sweta| 43|200000|  INDIA|     f|
|  13|   Raushan| 48|650000|    USA|     m|
|  14|    Mukesh| 36| 95000| RUSSIA|     m|
|  15|   Prakash| 52|750000|  INDIA|     m|
|1546|john_smith| 46| 48000|   NULL|     M|
+----+----------+---+------+-------+------+



1. Alias
2. where / filter
3. litaral
4. adding columns
5. renaming columns
6. casting datatypes
7. removing columns

1. Alias

In [0]:
new_employee_df.select(col('id').alias("emp_id"),'name','salary').show()

+------+----------+------+
|emp_id|      name|salary|
+------+----------+------+
|     1|    Manish| 75000|
|     2|    Nikita|100000|
|     3|    Pritam|150000|
|     4|  Prantosh|200000|
|     5|    Vikash|300000|
|     6|     Rahul|300000|
|     7|      Raju|540000|
|     8|   Praveen| 70000|
|     9|       Dev|150000|
|    10|    Sherin| 25000|
|    11|      Ragu| 35000|
|    12|     Sweta|200000|
|    13|   Raushan|650000|
|    14|    Mukesh| 95000|
|    15|   Prakash|750000|
|  1546|john_smith| 48000|
+------+----------+------+



2. where / filter -- both are same

In [0]:
new_employee_df.filter(col('salary')>150000).show()

+---+--------+---+------+-------+------+
| id|    name|age|salary|address|gender|
+---+--------+---+------+-------+------+
|  4|Prantosh| 17|200000|  JAPAN|     m|
|  5|  Vikash| 31|300000|    USA|     m|
|  6|   Rahul| 55|300000|  INDIA|     m|
|  7|    Raju| 67|540000|    USA|     m|
| 12|   Sweta| 43|200000|  INDIA|     f|
| 13| Raushan| 48|650000|    USA|     m|
| 15| Prakash| 52|750000|  INDIA|     m|
+---+--------+---+------+-------+------+



In [0]:
new_employee_df.where(col('salary')>150000).show()

+---+--------+---+------+-------+------+
| id|    name|age|salary|address|gender|
+---+--------+---+------+-------+------+
|  4|Prantosh| 17|200000|  JAPAN|     m|
|  5|  Vikash| 31|300000|    USA|     m|
|  6|   Rahul| 55|300000|  INDIA|     m|
|  7|    Raju| 67|540000|    USA|     m|
| 12|   Sweta| 43|200000|  INDIA|     f|
| 13| Raushan| 48|650000|    USA|     m|
| 15| Prakash| 52|750000|  INDIA|     m|
+---+--------+---+------+-------+------+



In [0]:
new_employee_df.where(col('salary')>150000 & col('age')<18).show()

---------------------------------------------------------------------------
PySparkValueError                         Traceback (most recent call last)
File <command-5622917355670394>, line 1
----> 1 new_employee_df.where(col('salary')>150000 & col('age')<18).show()

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/column.py:635, in Column.__nonzero__(self)
    634 def __nonzero__(self) -> None:
--> 635     raise PySparkValueError(
    636         errorClass="CANNOT_CONVERT_COLUMN_INTO_BOOL",
    637         messageParameters={},
    638     )

PySparkValueError: [CANNOT_CONVERT_COLUMN_INTO_BOOL] Cannot convert column into bool: please use '&' for 'and', '|' for 'or', '~' for 'not' when building DataFrame boolean expressions.

In [0]:
new_employee_df.where((col('salary')>150000) & (col('age')<18)).show()

+---+--------+---+------+-------+------+
| id|    name|age|salary|address|gender|
+---+--------+---+------+-------+------+
|  4|Prantosh| 17|200000|  JAPAN|     m|
+---+--------+---+------+-------+------+



3. litaral -- pass same value for all the records for a column(kind of default value)

* Creates a new projection (selects columns).
* We explicitly choose which columns to keep.
* Can add new columns while selecting existing ones.

In [0]:
new_employee_df.select('*',lit('kumar')).show()

+----+----------+---+------+-------+------+-----+
|  id|      name|age|salary|address|gender|kumar|
+----+----------+---+------+-------+------+-----+
|   1|    Manish| 26| 75000|  INDIA|     m|kumar|
|   2|    Nikita| 23|100000|    USA|     f|kumar|
|   3|    Pritam| 22|150000|  INDIA|     m|kumar|
|   4|  Prantosh| 17|200000|  JAPAN|     m|kumar|
|   5|    Vikash| 31|300000|    USA|     m|kumar|
|   6|     Rahul| 55|300000|  INDIA|     m|kumar|
|   7|      Raju| 67|540000|    USA|     m|kumar|
|   8|   Praveen| 28| 70000|  JAPAN|     m|kumar|
|   9|       Dev| 32|150000|  JAPAN|     m|kumar|
|  10|    Sherin| 16| 25000| RUSSIA|     f|kumar|
|  11|      Ragu| 12| 35000|  INDIA|     f|kumar|
|  12|     Sweta| 43|200000|  INDIA|     f|kumar|
|  13|   Raushan| 48|650000|    USA|     m|kumar|
|  14|    Mukesh| 36| 95000| RUSSIA|     m|kumar|
|  15|   Prakash| 52|750000|  INDIA|     m|kumar|
|1546|john_smith| 46| 48000|   NULL|     M|kumar|
+----+----------+---+------+-------+------+-----+


In [0]:
new_employee_df.printSchema()


root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- salary: integer (nullable = true)
 |-- address: string (nullable = true)
 |-- gender: string (nullable = true)



In [0]:
new_employee_df.select('*',lit('kumar').alias('last_name')).show()

+----+----------+---+------+-------+------+---------+
|  id|      name|age|salary|address|gender|last_name|
+----+----------+---+------+-------+------+---------+
|   1|    Manish| 26| 75000|  INDIA|     m|    kumar|
|   2|    Nikita| 23|100000|    USA|     f|    kumar|
|   3|    Pritam| 22|150000|  INDIA|     m|    kumar|
|   4|  Prantosh| 17|200000|  JAPAN|     m|    kumar|
|   5|    Vikash| 31|300000|    USA|     m|    kumar|
|   6|     Rahul| 55|300000|  INDIA|     m|    kumar|
|   7|      Raju| 67|540000|    USA|     m|    kumar|
|   8|   Praveen| 28| 70000|  JAPAN|     m|    kumar|
|   9|       Dev| 32|150000|  JAPAN|     m|    kumar|
|  10|    Sherin| 16| 25000| RUSSIA|     f|    kumar|
|  11|      Ragu| 12| 35000|  INDIA|     f|    kumar|
|  12|     Sweta| 43|200000|  INDIA|     f|    kumar|
|  13|   Raushan| 48|650000|    USA|     m|    kumar|
|  14|    Mukesh| 36| 95000| RUSSIA|     m|    kumar|
|  15|   Prakash| 52|750000|  INDIA|     m|    kumar|
|1546|john_smith| 46| 48000|

4. Adding columns

In [0]:
new_employee_df.withColumn('sur_name',lit('singh')).show()

# withColumn(columnName, colExpr)
# Adds or replaces a column.
# Returns a new DataFrame.
# Can be used to create a column based on existing columns.

+----+----------+---+------+-------+------+--------+
|  id|      name|age|salary|address|gender|sur_name|
+----+----------+---+------+-------+------+--------+
|   1|    Manish| 26| 75000|  INDIA|     m|   singh|
|   2|    Nikita| 23|100000|    USA|     f|   singh|
|   3|    Pritam| 22|150000|  INDIA|     m|   singh|
|   4|  Prantosh| 17|200000|  JAPAN|     m|   singh|
|   5|    Vikash| 31|300000|    USA|     m|   singh|
|   6|     Rahul| 55|300000|  INDIA|     m|   singh|
|   7|      Raju| 67|540000|    USA|     m|   singh|
|   8|   Praveen| 28| 70000|  JAPAN|     m|   singh|
|   9|       Dev| 32|150000|  JAPAN|     m|   singh|
|  10|    Sherin| 16| 25000| RUSSIA|     f|   singh|
|  11|      Ragu| 12| 35000|  INDIA|     f|   singh|
|  12|     Sweta| 43|200000|  INDIA|     f|   singh|
|  13|   Raushan| 48|650000|    USA|     m|   singh|
|  14|    Mukesh| 36| 95000| RUSSIA|     m|   singh|
|  15|   Prakash| 52|750000|  INDIA|     m|   singh|
|1546|john_smith| 46| 48000|   NULL|     M|   

In [0]:
new_employee_df.printSchema()

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- salary: integer (nullable = true)
 |-- address: string (nullable = true)
 |-- gender: string (nullable = true)



* Use withColumn() when adding/modifying one or two columns.
* Use select() when creating a new DataFrame with many derived columns or when choosing specific columns.

5. renaming columns (we can use alias or withColumnRenamed())

In [0]:
new_employee_df.withColumnRenamed('id','employee_id').show()

+-----------+----------+---+------+-------+------+
|employee_id|      name|age|salary|address|gender|
+-----------+----------+---+------+-------+------+
|          1|    Manish| 26| 75000|  INDIA|     m|
|          2|    Nikita| 23|100000|    USA|     f|
|          3|    Pritam| 22|150000|  INDIA|     m|
|          4|  Prantosh| 17|200000|  JAPAN|     m|
|          5|    Vikash| 31|300000|    USA|     m|
|          6|     Rahul| 55|300000|  INDIA|     m|
|          7|      Raju| 67|540000|    USA|     m|
|          8|   Praveen| 28| 70000|  JAPAN|     m|
|          9|       Dev| 32|150000|  JAPAN|     m|
|         10|    Sherin| 16| 25000| RUSSIA|     f|
|         11|      Ragu| 12| 35000|  INDIA|     f|
|         12|     Sweta| 43|200000|  INDIA|     f|
|         13|   Raushan| 48|650000|    USA|     m|
|         14|    Mukesh| 36| 95000| RUSSIA|     m|
|         15|   Prakash| 52|750000|  INDIA|     m|
|       1546|john_smith| 46| 48000|   NULL|     M|
+-----------+----------+---+---

Saving the dataframe

In [0]:
new_employee_df2 = new_employee_df.withColumnRenamed('id','employee_id')
new_employee_df2.show()

+-----------+----------+---+------+-------+------+
|employee_id|      name|age|salary|address|gender|
+-----------+----------+---+------+-------+------+
|          1|    Manish| 26| 75000|  INDIA|     m|
|          2|    Nikita| 23|100000|    USA|     f|
|          3|    Pritam| 22|150000|  INDIA|     m|
|          4|  Prantosh| 17|200000|  JAPAN|     m|
|          5|    Vikash| 31|300000|    USA|     m|
|          6|     Rahul| 55|300000|  INDIA|     m|
|          7|      Raju| 67|540000|    USA|     m|
|          8|   Praveen| 28| 70000|  JAPAN|     m|
|          9|       Dev| 32|150000|  JAPAN|     m|
|         10|    Sherin| 16| 25000| RUSSIA|     f|
|         11|      Ragu| 12| 35000|  INDIA|     f|
|         12|     Sweta| 43|200000|  INDIA|     f|
|         13|   Raushan| 48|650000|    USA|     m|
|         14|    Mukesh| 36| 95000| RUSSIA|     m|
|         15|   Prakash| 52|750000|  INDIA|     m|
|       1546|john_smith| 46| 48000|   NULL|     M|
+-----------+----------+---+---

6. Casting

In [0]:
new_employee_df.printSchema()

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- salary: integer (nullable = true)
 |-- address: string (nullable = true)
 |-- gender: string (nullable = true)



In [0]:
new_employee_df3 = new_employee_df.withColumn('id',col('id').cast('string'))
new_employee_df3.printSchema()

root
 |-- id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- salary: integer (nullable = true)
 |-- address: string (nullable = true)
 |-- gender: string (nullable = true)



In [0]:
new_employee_df4 = new_employee_df.withColumn('id',col('id').cast('string'))\
    .withColumn('salary',col('salary').cast('long'))
new_employee_df4.printSchema()

root
 |-- id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- salary: long (nullable = true)
 |-- address: string (nullable = true)
 |-- gender: string (nullable = true)



7. removing columns

In [0]:
new_employee_df5 = new_employee_df.drop('id',col('gender'))
new_employee_df5.printSchema()

root
 |-- name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- salary: integer (nullable = true)
 |-- address: string (nullable = true)



using Spark sql

In [0]:

new_employee_df.createTempView('employee_view')

In [0]:
spark.sql("""select * from employee_view where salary>150000 and age<18; """).show()

+---+--------+---+------+-------+------+
| id|    name|age|salary|address|gender|
+---+--------+---+------+-------+------+
|  4|Prantosh| 17|200000|  JAPAN|     m|
+---+--------+---+------+-------+------+



In [0]:
spark.sql("""select *,'kumar' as last_name from employee_view where salary>150000 and age<18; """).show()

+---+--------+---+------+-------+------+---------+
| id|    name|age|salary|address|gender|last_name|
+---+--------+---+------+-------+------+---------+
|  4|Prantosh| 17|200000|  JAPAN|     m|    kumar|
+---+--------+---+------+-------+------+---------+



In [0]:
spark.sql("""select *,'kumar' as last_name,concat(name,' ',last_name)as full_name 
           from employee_view where salary>150000 and age<18; """).show()

+---+--------+---+------+-------+------+---------+--------------+
| id|    name|age|salary|address|gender|last_name|     full_name|
+---+--------+---+------+-------+------+---------+--------------+
|  4|Prantosh| 17|200000|  JAPAN|     m|    kumar|Prantosh kumar|
+---+--------+---+------+-------+------+---------+--------------+



In [0]:
spark.sql("""select *,'kumar' as last_name,concat(name,' ',last_name)as full_name,id as emp_id 
           from employee_view where salary>150000 and age<18; """).show()

+---+--------+---+------+-------+------+---------+--------------+------+
| id|    name|age|salary|address|gender|last_name|     full_name|emp_id|
+---+--------+---+------+-------+------+---------+--------------+------+
|  4|Prantosh| 17|200000|  JAPAN|     m|    kumar|Prantosh kumar|     4|
+---+--------+---+------+-------+------+---------+--------------+------+



In [0]:
spark.sql("""select *,'kumar' as last_name,concat(name,' ',last_name)as full_name,cast(id as string) 
           from employee_view where salary>150000 and age<18; """).printSchema()

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- salary: integer (nullable = true)
 |-- address: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- last_name: string (nullable = false)
 |-- full_name: string (nullable = true)
 |-- id: string (nullable = true)



In [0]:
# removing column
# dont select those columns that are not needed

Union and unionAll unionByName
* In dataframe no differance between union and unionAll, but sql both are different Union gives distinct and unionAll gives all records.
*
*
*

In [0]:
my_data = [(10,'Anil',50000,18),
        (11,'vikas',75000,16),
        (12,'Nisha',45000,18),
        (13,'Priya',58000,17),
        (14,'Mohit',50700,18),
        (15,'Rajesh',80000,18),
        (16,'sam',44000,20),
        (17,'sam',47000,16),
        (16,'sam',44000,20)]


my_schema = ['id','name','sal','mngr_id']

manager_df = spark.createDataFrame(data=my_data,schema=my_schema)
manager_df.show()

+---+------+-----+-------+
| id|  name|  sal|mngr_id|
+---+------+-----+-------+
| 10|  Anil|50000|     18|
| 11| vikas|75000|     16|
| 12| Nisha|45000|     18|
| 13| Priya|58000|     17|
| 14| Mohit|50700|     18|
| 15|Rajesh|80000|     18|
| 16|   sam|44000|     20|
| 17|   sam|47000|     16|
| 16|   sam|44000|     20|
+---+------+-----+-------+



In [0]:
data1 = [(19,'Sudip',44000,25),(20,'Rahul',80000,18)]
schema1 = ['id','name','sal','mngr_id']

maneger_df1 = spark.createDataFrame(data=data1,schema=schema1)
maneger_df1.show()

+---+-----+-----+-------+
| id| name|  sal|mngr_id|
+---+-----+-----+-------+
| 19|Sudip|44000|     25|
| 20|Rahul|80000|     18|
+---+-----+-----+-------+



In [0]:
manager_df.union(maneger_df1).show()

+---+------+-----+-------+
| id|  name|  sal|mngr_id|
+---+------+-----+-------+
| 10|  Anil|50000|     18|
| 11| vikas|75000|     16|
| 12| Nisha|45000|     18|
| 13| Priya|58000|     17|
| 14| Mohit|50700|     18|
| 15|Rajesh|80000|     18|
| 16|   sam|44000|     20|
| 17|   sam|47000|     16|
| 16|   sam|44000|     20|
| 19| Sudip|44000|     25|
| 20| Rahul|80000|     18|
+---+------+-----+-------+



In [0]:
manager_df.union(maneger_df1).count()

11

In [0]:
manager_df.unionAll(maneger_df1).show()

+---+------+-----+-------+
| id|  name|  sal|mngr_id|
+---+------+-----+-------+
| 10|  Anil|50000|     18|
| 11| vikas|75000|     16|
| 12| Nisha|45000|     18|
| 13| Priya|58000|     17|
| 14| Mohit|50700|     18|
| 15|Rajesh|80000|     18|
| 16|   sam|44000|     20|
| 17|   sam|47000|     16|
| 16|   sam|44000|     20|
| 19| Sudip|44000|     25|
| 20| Rahul|80000|     18|
+---+------+-----+-------+



In [0]:
manager_df.unionAll(maneger_df1).count()

11

In [0]:
wrong_data1 = [(19,'Sudip',25,44000),(20,'Rahul',18,80000)]
wrong_schema1 = ['id','name','mngr_id','sal']

wrong_maneger_df1 = spark.createDataFrame(data=wrong_data1,schema=wrong_schema1)
wrong_maneger_df1.show()

+---+-----+-------+-----+
| id| name|mngr_id|  sal|
+---+-----+-------+-----+
| 19|Sudip|     25|44000|
| 20|Rahul|     18|80000|
+---+-----+-------+-----+



In [0]:
manager_df.unionAll(wrong_maneger_df1).show()

+---+------+-----+-------+
| id|  name|  sal|mngr_id|
+---+------+-----+-------+
| 10|  Anil|50000|     18|
| 11| vikas|75000|     16|
| 12| Nisha|45000|     18|
| 13| Priya|58000|     17|
| 14| Mohit|50700|     18|
| 15|Rajesh|80000|     18|
| 16|   sam|44000|     20|
| 17|   sam|47000|     16|
| 16|   sam|44000|     20|
| 19| Sudip|   25|  44000|
| 20| Rahul|   18|  80000|
+---+------+-----+-------+



In [0]:
manager_df.unionByName(wrong_maneger_df1).show()

+---+------+-----+-------+
| id|  name|  sal|mngr_id|
+---+------+-----+-------+
| 10|  Anil|50000|     18|
| 11| vikas|75000|     16|
| 12| Nisha|45000|     18|
| 13| Priya|58000|     17|
| 14| Mohit|50700|     18|
| 15|Rajesh|80000|     18|
| 16|   sam|44000|     20|
| 17|   sam|47000|     16|
| 16|   sam|44000|     20|
| 19| Sudip|44000|     25|
| 20| Rahul|80000|     18|
+---+------+-----+-------+



In [0]:
wrong_data2 = [(19,'Sudip',25,44000),(20,'Rahul',18,80000)]
wrong_schema2 = ['id','name','mngr_id0','sal0']

wrong_maneger_df2 = spark.createDataFrame(data=wrong_data2,schema=wrong_schema2)
wrong_maneger_df2.show()

+---+-----+--------+-----+
| id| name|mngr_id0| sal0|
+---+-----+--------+-----+
| 19|Sudip|      25|44000|
| 20|Rahul|      18|80000|
+---+-----+--------+-----+



In [0]:
manager_df.unionByName(wrong_maneger_df2).show()

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-6967008125337949>, line 1
----> 1 manager_df.unionByName(wrong_maneger_df2).show()

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:1156, in DataFrame.show(self, n, truncate, vertical)
   1155 def show(self, n: int = 20, truncate: Union[bool, int] = True, vertical: bool = False) -> None:
-> 1156     print(self._show_string(n, truncate, vertical))

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:909, in DataFrame._show_string(self, n, truncate, vertical)
    892     except ValueError:
    893         raise PySparkTypeError(
    894             errorClass="NOT_BOOL",
    895             messageParameters={
   (...)
    898             },
    899         )
    901 table, _ = DataFrame(
    902     plan.ShowString(
    903         child=self._plan,
   

In [0]:
wrong_data3 = [(19,'Sudip',25,44000,30),(20,'Rahul',18,80000,45)]
wrong_schema3 = ['id','name','mngr_id','sal','age']

wrong_maneger_df3 = spark.createDataFrame(data=wrong_data3,schema=wrong_schema3)
wrong_maneger_df3.show()

+---+-----+-------+-----+---+
| id| name|mngr_id|  sal|age|
+---+-----+-------+-----+---+
| 19|Sudip|     25|44000| 30|
| 20|Rahul|     18|80000| 45|
+---+-----+-------+-----+---+



In [0]:
manager_df.unionByName(wrong_maneger_df3).show()

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-6967008125337951>, line 1
----> 1 manager_df.unionByName(wrong_maneger_df3).show()

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:1156, in DataFrame.show(self, n, truncate, vertical)
   1155 def show(self, n: int = 20, truncate: Union[bool, int] = True, vertical: bool = False) -> None:
-> 1156     print(self._show_string(n, truncate, vertical))

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:909, in DataFrame._show_string(self, n, truncate, vertical)
    892     except ValueError:
    893         raise PySparkTypeError(
    894             errorClass="NOT_BOOL",
    895             messageParameters={
   (...)
    898             },
    899         )
    901 table, _ = DataFrame(
    902     plan.ShowString(
    903         child=self._plan,
   

In [0]:
manager_df.unionByName(wrong_maneger_df3.select('id','name','mngr_id','sal')).show()

+---+------+-----+-------+
| id|  name|  sal|mngr_id|
+---+------+-----+-------+
| 10|  Anil|50000|     18|
| 11| vikas|75000|     16|
| 12| Nisha|45000|     18|
| 13| Priya|58000|     17|
| 14| Mohit|50700|     18|
| 15|Rajesh|80000|     18|
| 16|   sam|44000|     20|
| 17|   sam|47000|     16|
| 16|   sam|44000|     20|
| 19| Sudip|44000|     25|
| 20| Rahul|80000|     18|
+---+------+-----+-------+



repartition() and coalesce():

* repartition() and coalesce() are both used to change the number of partitions in a Spark DataFrame, but they work differently and are used in different scenarios.


| Feature             | repartition()        | coalesce()      |
| ------------------- | ------------------   | -------------   |
| Increase partitions | ✅ Yes               | ❌ No          |
| Decrease partitions | ✅ Yes               | ✅ Yes         |
| Shuffle data        | ✅ Always            | ❌ Usually No  |
| Performance cost    | Expensive            | Cheap           |
| Data distribution   | Evenly distributed   | May be uneven   |



* repartition = Redistribute data (shuffle)

* coalesce    = Combine partitions (no shuffle)

| Scenario                                | Use                      | Why                           | What NOT to Use   | Alternative              |
| --------------------------------------- | ------------------------ | ----------------------------- | ----------------- | ------------------------ |
| Increase partitions (10 → 100)          | `repartition(100)`       | Increases parallelism         | `coalesce(100)`   | `repartition(100)`       |
| Decrease partitions (100 → 10)          | `coalesce(10)`           | Avoids expensive shuffle      | `repartition(10)` | `coalesce(10)`           |
| Before large joins                      | `repartition(join_key)`  | Better data distribution      | `coalesce()`      | `repartition()`          |
| Before `groupBy()` on huge data         | `repartition(group_key)` | Reduces skew                  | `coalesce()`      | `repartition()`          |
| Fix skewed partitions                   | `repartition()`          | Redistributes data evenly     | `coalesce()`      | `repartition()`          |
| Reduce output files                     | `coalesce()`             | Fast, no full shuffle         | `repartition()`   | `coalesce()`             |
| Create single output file               | `coalesce(1)`            | Merges partitions efficiently | `repartition(1)`  | `coalesce(1)`            |
| Data already well distributed           | Do nothing               | Avoid unnecessary shuffle     | `repartition()`   | Keep existing partitions |
| Small dataset, writing output           | `coalesce()`             | Lower overhead                | `repartition()`   | `coalesce()`             |
| Need balanced partitions across cluster | `repartition()`          | Evenly redistributes records  | `coalesce()`      | `repartition()`          |




Increases parallelism means Spark can execute more tasks simultaneously across the cluster.

Example

Suppose we have:

df.rdd.getNumPartitions() = 4

And our cluster has:

8 CPU cores

Spark can run at most 4 tasks at the same time because there are only 4 partitions.

Partition 1 → Core 1;
Partition 2 → Core 2;
Partition 3 → Core 3;
Partition 4 → Core 4;

Core 5, 6, 7, 8 are idle


Increase partitions
df = df.repartition(16)

Now:

df.rdd.getNumPartitions() = 16

Spark can run up to 8 tasks simultaneously (limited by available cores).

Partitions 1-8  → Run together
Partitions 9-16 → Run next means after Partitions 1-8

All 8 cores are utilized.

In [0]:
flight_df = spark.read.format("csv")\
          .option("header", "true")\
          .option("inferschema", "true")\
          .option('mode','FAILFAST')\
          .load("/Volumes/workspace/default/data_files/2015-summary.csv")
flight_df.show(5)

+-----------------+-------------------+-----+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|
+-----------------+-------------------+-----+
|    United States|            Romania|   15|
|    United States|            Croatia|    1|
|    United States|            Ireland|  344|
|            Egypt|      United States|   15|
|    United States|              India|   62|
+-----------------+-------------------+-----+
only showing top 5 rows


In [0]:
flight_df.rdd.getNumPartitions()

---------------------------------------------------------------------------
PySparkNotImplementedError                Traceback (most recent call last)
File <command-6198640188654547>, line 1
----> 1 flight_df.rdd.getNumPartitions()

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:2373, in DataFrame.rdd(self)
   2371 @property
   2372 def rdd(self) -> "RDD[Row]":
-> 2373     raise PySparkNotImplementedError(
   2374         errorClass="NOT_IMPLEMENTED",
   2375         messageParameters={"feature": "rdd"},
   2376     )

PySparkNotImplementedError: [NOT_IMPLEMENTED] Using custom code using PySpark RDDs is not allowed on serverless compute. We suggest using mapInPandas or mapInArrow for the most common use cases, or switch to Dedicated access mode if you require RDDs. For more details on compatibility and limitations, check: https://docs.databricks.com/release-notes/serverless.html#limitations

In [0]:
from pyspark.sql.functions import spark_partition_id

flight_df.select(spark_partition_id()).distinct().count() # 128 mb for one partition

1

In [0]:
partition_flight_df.withColumn('partition_id',spark_partition_id()).groupBy('partition_id').count().show()

+------------+-----+
|partition_id|count|
+------------+-----+
|           0|   51|
|           1|   51|
|           2|   51|
|           3|   51|
|           4|   52|
+------------+-----+



In [0]:
partition_flight_df.select(spark_partition_id()).distinct().count() # 128 mb for one partition

5

In [0]:
partition_flight_df = flight_df.repartition(5)

In [0]:
partition_flight_df_300 = flight_df.repartition(300,'ORIGIN_COUNTRY_NAME')

In [0]:
partition_flight_df_300.select(spark_partition_id()).count() # 128 mb for one partition

256

In [0]:
partition_flight_df_300.withColumn('partition_id',spark_partition_id()).groupBy('partition_id').count().show(300)

+------------+-----+
|partition_id|count|
+------------+-----+
|           0|    1|
|           2|    2|
|           7|    1|
|          10|    1|
|          13|    1|
|          15|    2|
|          16|    2|
|          19|    2|
|          22|    1|
|          28|    1|
|          31|    1|
|          39|    1|
|          43|    1|
|          44|    1|
|          45|    2|
|          48|    1|
|          53|    1|
|          55|    1|
|          65|    1|
|          70|    1|
|          73|    2|
|          75|    1|
|          76|    1|
|          81|    1|
|          84|    2|
|          86|    1|
|          87|    1|
|          90|    1|
|          91|    1|
|          97|    2|
|         100|    1|
|         103|    2|
|         104|    1|
|         108|    1|
|         112|    2|
|         115|    1|
|         117|    2|
|         126|    1|
|         127|    2|
|         129|    1|
|         130|    2|
|         132|    1|
|         133|    1|
|         138|    2|
|         144

In [0]:
colease_partitions = flight_df.repartition(8)

In [0]:
colease_partitions_df = colease_partitions.coalesce(3)

In [0]:
colease_partitions_df.withColumn('partition_id',spark_partition_id()).groupBy('partition_id').count().show()

+------------+-----+
|partition_id|count|
+------------+-----+
|           0|   64|
|           1|   96|
|           2|   96|
+------------+-----+



In [0]:
re_partitions_df_3 = flight_df.repartition(3)

In [0]:
re_partitions_df_3.withColumn('partition_id',spark_partition_id()).groupBy('partition_id').count().show()

+------------+-----+
|partition_id|count|
+------------+-----+
|           0|   86|
|           1|   85|
|           2|   85|
+------------+-----+



In [0]:
re_partitions_df_8 = flight_df.repartition(8)

In [0]:
coalece_partitions  = re_partitions_df_8.coalesce(10)

In [0]:
coalece_partitions.coalesce(10).withColumn('partition_id',spark_partition_id()).groupBy('partition_id').count().show()

+------------+-----+
|partition_id|count|
+------------+-----+
|           0|   32|
|           1|   32|
|           2|   32|
|           3|   32|
|           4|   32|
|           5|   32|
|           6|   32|
|           7|   32|
+------------+-----+



Notes :
* repartition gives evenly distributed, and coalesce gives uneven partitions.
* performing repartition suffles the data so it is expensive opereation, and coalesce does not suffles the data so it is less expensive operation.
* coalesce only marge partitions , never increase partitions only decrease. 

Conditional statements and creaating columns based on that


In [0]:
from pyspark.sql import Row
employee_df = spark.read \
    .format("csv") \
    .option("inferSchema", "true")\
    .option("header", "true") \
    .load("/Volumes/workspace/default/data_files/employee_data.csv")

new_row = [Row(1546, "john_smith", 46, 48000, None, "M")]

row_df = spark.createDataFrame(new_row, schema=employee_df.schema)

new_employee_df = employee_df.union(row_df)

display(new_employee_df)

id,name,age,salary,address,gender
1,Manish,26,75000,INDIA,m
2,Nikita,23,100000,USA,f
3,Pritam,22,150000,INDIA,m
4,Prantosh,17,200000,JAPAN,m
5,Vikash,31,300000,USA,m
6,Rahul,55,300000,INDIA,m
7,Raju,67,540000,USA,m
8,Praveen,28,70000,JAPAN,m
9,Dev,32,150000,JAPAN,m
10,Sherin,16,25000,RUSSIA,f


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

new_employee_df.withColumn('is_address',when(col('address') =='INDIA','local')
    .when(col('address') !='INDIA','foreign')
    .otherwise('No value')).show()


+----+----------+---+------+-------+------+----------+
|  id|      name|age|salary|address|gender|is_address|
+----+----------+---+------+-------+------+----------+
|   1|    Manish| 26| 75000|  INDIA|     m|     local|
|   2|    Nikita| 23|100000|    USA|     f|   foreign|
|   3|    Pritam| 22|150000|  INDIA|     m|     local|
|   4|  Prantosh| 17|200000|  JAPAN|     m|   foreign|
|   5|    Vikash| 31|300000|    USA|     m|   foreign|
|   6|     Rahul| 55|300000|  INDIA|     m|     local|
|   7|      Raju| 67|540000|    USA|     m|   foreign|
|   8|   Praveen| 28| 70000|  JAPAN|     m|   foreign|
|   9|       Dev| 32|150000|  JAPAN|     m|   foreign|
|  10|    Sherin| 16| 25000| RUSSIA|     f|   foreign|
|  11|      Ragu| 12| 35000|  INDIA|     f|     local|
|  12|     Sweta| 43|200000|  INDIA|     f|     local|
|  13|   Raushan| 48|650000|    USA|     m|   foreign|
|  14|    Mukesh| 36| 95000| RUSSIA|     m|   foreign|
|  15|   Prakash| 52|750000|  INDIA|     m|     local|
|1546|john

In [0]:

from pyspark.sql.functions import *
from pyspark.sql.types import *

new_employee_df.withColumn('address',when(col('address').isNull(),lit('UK'))
                           .otherwise(col('address')))\
                           .withColumn('is_address',when(col('address') =='INDIA','local')
                            .otherwise('foreign')).show()


+----+----------+---+------+-------+------+----------+
|  id|      name|age|salary|address|gender|is_address|
+----+----------+---+------+-------+------+----------+
|   1|    Manish| 26| 75000|  INDIA|     m|     local|
|   2|    Nikita| 23|100000|    USA|     f|   foreign|
|   3|    Pritam| 22|150000|  INDIA|     m|     local|
|   4|  Prantosh| 17|200000|  JAPAN|     m|   foreign|
|   5|    Vikash| 31|300000|    USA|     m|   foreign|
|   6|     Rahul| 55|300000|  INDIA|     m|     local|
|   7|      Raju| 67|540000|    USA|     m|   foreign|
|   8|   Praveen| 28| 70000|  JAPAN|     m|   foreign|
|   9|       Dev| 32|150000|  JAPAN|     m|   foreign|
|  10|    Sherin| 16| 25000| RUSSIA|     f|   foreign|
|  11|      Ragu| 12| 35000|  INDIA|     f|     local|
|  12|     Sweta| 43|200000|  INDIA|     f|     local|
|  13|   Raushan| 48|650000|    USA|     m|   foreign|
|  14|    Mukesh| 36| 95000| RUSSIA|     m|   foreign|
|  15|   Prakash| 52|750000|  INDIA|     m|     local|
|1546|john

Unique short and other functionality.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
data=[(10 ,'Anil',50000, 18),
(11 ,'Vikas',75000,  16),
(12 ,'Nisha',40000,  18),
(13 ,'Nidhi',60000,  17),
(14 ,'Priya',80000,  18),
(15 ,'Mohit',45000,  18),
(16 ,'Rajesh',90000, 10),
(17 ,'Raman',55000, 16),
(18 ,'Sam',65000,   17),
(15 ,'Mohit',45000,  18),
(13 ,'Nidhi',60000,  17),      
(14 ,'Priya',90000,  18),  
(18 ,'Sam',65000,   17)
     ]
our_schema = StructType([
    StructField('id', IntegerType(), True),
    StructField('name', StringType(), True),
    StructField('salary', IntegerType(), True),
    StructField('mngr_id', IntegerType(), True)
])

maneger_df = spark.createDataFrame(data=data, schema=our_schema)
maneger_df.show()

+---+------+------+-------+
| id|  name|salary|mngr_id|
+---+------+------+-------+
| 10|  Anil| 50000|     18|
| 11| Vikas| 75000|     16|
| 12| Nisha| 40000|     18|
| 13| Nidhi| 60000|     17|
| 14| Priya| 80000|     18|
| 15| Mohit| 45000|     18|
| 16|Rajesh| 90000|     10|
| 17| Raman| 55000|     16|
| 18|   Sam| 65000|     17|
| 15| Mohit| 45000|     18|
| 13| Nidhi| 60000|     17|
| 14| Priya| 90000|     18|
| 18|   Sam| 65000|     17|
+---+------+------+-------+



In [0]:
# show only distinct rows
maneger_df.distinct().show()

+---+------+------+-------+
| id|  name|salary|mngr_id|
+---+------+------+-------+
| 10|  Anil| 50000|     18|
| 11| Vikas| 75000|     16|
| 12| Nisha| 40000|     18|
| 13| Nidhi| 60000|     17|
| 14| Priya| 80000|     18|
| 15| Mohit| 45000|     18|
| 16|Rajesh| 90000|     10|
| 17| Raman| 55000|     16|
| 18|   Sam| 65000|     17|
| 14| Priya| 90000|     18|
+---+------+------+-------+



In [0]:
# show only distinct rows based on columns
maneger_df.distinct('id','name').show()

---------------------------------------------------------------------------
TypeError                                 Traceback (most recent call last)
File <command-5114330050736284>, line 2
      1 # show only distinct rows
----> 2 maneger_df.distinct('id','name').show()

TypeError: DataFrame.distinct() takes 1 positional argument but 3 were given

In [0]:
# show only distinct rows based on columns
maneger_df.select('id','name').distinct().show()

+---+------+
| id|  name|
+---+------+
| 10|  Anil|
| 11| Vikas|
| 12| Nisha|
| 13| Nidhi|
| 14| Priya|
| 15| Mohit|
| 16|Rajesh|
| 17| Raman|
| 18|   Sam|
+---+------+



In [0]:
maneger_df.dropDuplicates().show()

+---+------+------+-------+
| id|  name|salary|mngr_id|
+---+------+------+-------+
| 10|  Anil| 50000|     18|
| 11| Vikas| 75000|     16|
| 12| Nisha| 40000|     18|
| 13| Nidhi| 60000|     17|
| 14| Priya| 80000|     18|
| 15| Mohit| 45000|     18|
| 16|Rajesh| 90000|     10|
| 17| Raman| 55000|     16|
| 18|   Sam| 65000|     17|
| 14| Priya| 90000|     18|
+---+------+------+-------+



In [0]:
maneger_df.drop_duplicates(['id','name']).show()

+---+------+------+-------+
| id|  name|salary|mngr_id|
+---+------+------+-------+
| 10|  Anil| 50000|     18|
| 11| Vikas| 75000|     16|
| 12| Nisha| 40000|     18|
| 13| Nidhi| 60000|     17|
| 14| Priya| 80000|     18|
| 15| Mohit| 45000|     18|
| 16|Rajesh| 90000|     10|
| 17| Raman| 55000|     16|
| 18|   Sam| 65000|     17|
+---+------+------+-------+



In [0]:
# sorting the data
maneger_df.sort(col('salary')).show()

+---+------+------+-------+
| id|  name|salary|mngr_id|
+---+------+------+-------+
| 12| Nisha| 40000|     18|
| 15| Mohit| 45000|     18|
| 15| Mohit| 45000|     18|
| 10|  Anil| 50000|     18|
| 17| Raman| 55000|     16|
| 13| Nidhi| 60000|     17|
| 13| Nidhi| 60000|     17|
| 18|   Sam| 65000|     17|
| 18|   Sam| 65000|     17|
| 11| Vikas| 75000|     16|
| 14| Priya| 80000|     18|
| 16|Rajesh| 90000|     10|
| 14| Priya| 90000|     18|
+---+------+------+-------+



In [0]:
# sorting the data in desc
maneger_df.sort(col('salary').desc()).show()

+---+------+------+-------+
| id|  name|salary|mngr_id|
+---+------+------+-------+
| 16|Rajesh| 90000|     10|
| 14| Priya| 90000|     18|
| 14| Priya| 80000|     18|
| 11| Vikas| 75000|     16|
| 18|   Sam| 65000|     17|
| 18|   Sam| 65000|     17|
| 13| Nidhi| 60000|     17|
| 13| Nidhi| 60000|     17|
| 17| Raman| 55000|     16|
| 10|  Anil| 50000|     18|
| 15| Mohit| 45000|     18|
| 15| Mohit| 45000|     18|
| 12| Nisha| 40000|     18|
+---+------+------+-------+



In [0]:
# sorting the data in desc on multiple columns
maneger_df.sort(col('salary').desc(),col('name').desc()).show()

+---+------+------+-------+
| id|  name|salary|mngr_id|
+---+------+------+-------+
| 16|Rajesh| 90000|     10|
| 14| Priya| 90000|     18|
| 14| Priya| 80000|     18|
| 11| Vikas| 75000|     16|
| 18|   Sam| 65000|     17|
| 18|   Sam| 65000|     17|
| 13| Nidhi| 60000|     17|
| 13| Nidhi| 60000|     17|
| 17| Raman| 55000|     16|
| 10|  Anil| 50000|     18|
| 15| Mohit| 45000|     18|
| 15| Mohit| 45000|     18|
| 12| Nisha| 40000|     18|
+---+------+------+-------+



In [0]:
leet_code_data = [
    (1, 'Will', None),
    (2, 'Jane', None),
    (3, 'Alex', 2),
    (4, 'Bill', None),
    (5, 'Zack', 1),
    (6, 'Mark', 2)
]



agrigation


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
emp_data = [
(1,'manish',26,20000,'india','IT'),
(2,'rahul',None,40000,'germany','engineering'),
(3,'pawan',12,60000,'india','sales'),
(4,'roshini',44,None,'uk','engineering'),
(5,'raushan',35,70000,'india','sales'),
(6,None,29,200000,'uk','IT'),
(7,'adam',37,65000,'us','IT'),
(8,'chris',16,40000,'us','sales'),
(None,None,None,None,None,None),
(7,'adam',37,65000,'us','IT')
]

employee_df = spark.createDataFrame(data = emp_data,schema=['emp_id','name','age','salary','country','department'])
employee_df.show()

+------+-------+----+------+-------+-----------+
|emp_id|   name| age|salary|country| department|
+------+-------+----+------+-------+-----------+
|     1| manish|  26| 20000|  india|         IT|
|     2|  rahul|NULL| 40000|germany|engineering|
|     3|  pawan|  12| 60000|  india|      sales|
|     4|roshini|  44|  NULL|     uk|engineering|
|     5|raushan|  35| 70000|  india|      sales|
|     6|   NULL|  29|200000|     uk|         IT|
|     7|   adam|  37| 65000|     us|         IT|
|     8|  chris|  16| 40000|     us|      sales|
|  NULL|   NULL|NULL|  NULL|   NULL|       NULL|
|     7|   adam|  37| 65000|     us|         IT|
+------+-------+----+------+-------+-----------+



In [0]:
employee_df.count()

10

In [0]:
employee_df.select(count('name')).show()

+-----------+
|count(name)|
+-----------+
|          8|
+-----------+



In [0]:
employee_df.select(count('*')).show()

+--------+
|count(1)|
+--------+
|      10|
+--------+



In [0]:
employee_df.select(sum('salary').alias('total_salary'),max('salary').alias('maximum_salary'),
                   min('salary').alias('minimum_salary')).show()

+------------+--------------+--------------+
|total_salary|maximum_salary|minimum_salary|
+------------+--------------+--------------+
|      560000|        200000|         20000|
+------------+--------------+--------------+



In [0]:
employee_df.select(sum('salary').alias('total_salary'),count('salary').alias('count_of_salary'),
                   avg('salary').alias('average_salary').cast('int')).show()

+------------+---------------+--------------+
|total_salary|count_of_salary|average_salary|
+------------+---------------+--------------+
|      560000|              8|         70000|
+------------+---------------+--------------+



Group by

In [0]:
emp_df = [(1,'manish',50000,'IT'),
(2,'vikash',60000,'sales'),
(3,'raushan',70000,'marketing'),
(4,'mukesh',80000,'IT'),
(5,'pritam',90000,'sales'),
(6,'nikita',45000,'marketing'),
(7,'ragini',55000,'marketing'),
(8,'rakesh',100000,'IT'),
(9,'aditya',65000,'IT'),
(10,'rahul',50000,'marketing')]


employee_df = spark.createDataFrame(data = emp_df,schema=['emp_id','name','salary','department'])
employee_df.show()

+------+-------+------+----------+
|emp_id|   name|salary|department|
+------+-------+------+----------+
|     1| manish| 50000|        IT|
|     2| vikash| 60000|     sales|
|     3|raushan| 70000| marketing|
|     4| mukesh| 80000|        IT|
|     5| pritam| 90000|     sales|
|     6| nikita| 45000| marketing|
|     7| ragini| 55000| marketing|
|     8| rakesh|100000|        IT|
|     9| aditya| 65000|        IT|
|    10|  rahul| 50000| marketing|
+------+-------+------+----------+



In [0]:
employee_df.groupBy('department').agg(sum('salary')).show()

+----------+-----------+
|department|sum(salary)|
+----------+-----------+
|        IT|     295000|
|     sales|     150000|
| marketing|     220000|
+----------+-----------+



In [0]:
emp_data=[(1,'manish',50000,'IT','india'),
(2,'vikash',60000,'sales','us'),
(3,'raushan',70000,'marketing','india'),
(4,'mukesh',80000,'IT','us'),
(5,'pritam',90000,'sales','india'),
(6,'nikita',45000,'marketing','us'),
(7,'ragini',55000,'marketing','india'),
(8,'rakesh',100000,'IT','us'),
(9,'aditya',65000,'IT','india'),
(10,'rahul',50000,'marketing','us')]

employee_df2 = spark.createDataFrame(data = emp_data,schema=['emp_id','name','salary','department','country'])
employee_df2.show()

+------+-------+------+----------+-------+
|emp_id|   name|salary|department|country|
+------+-------+------+----------+-------+
|     1| manish| 50000|        IT|  india|
|     2| vikash| 60000|     sales|     us|
|     3|raushan| 70000| marketing|  india|
|     4| mukesh| 80000|        IT|     us|
|     5| pritam| 90000|     sales|  india|
|     6| nikita| 45000| marketing|     us|
|     7| ragini| 55000| marketing|  india|
|     8| rakesh|100000|        IT|     us|
|     9| aditya| 65000|        IT|  india|
|    10|  rahul| 50000| marketing|     us|
+------+-------+------+----------+-------+



In [0]:
employee_df2.groupBy("department","country").agg(sum("salary").alias("total_salary")).orderBy('country',"department").show()

+----------+-------+------------+
|department|country|total_salary|
+----------+-------+------------+
|        IT|  india|      115000|
| marketing|  india|      125000|
|     sales|  india|       90000|
|        IT|     us|      180000|
| marketing|     us|       95000|
|     sales|     us|       60000|
+----------+-------+------------+



In [0]:

customer_data = [(1,'manish','patna',"30-05-2022"),
(2,'vikash','kolkata',"12-03-2023"),
(3,'nikita','delhi',"25-06-2023"),
(4,'rahul','ranchi',"24-03-2023"),
(5,'mahesh','jaipur',"22-03-2023"),
(6,'prantosh','kolkata',"18-10-2022"),
(7,'raman','patna',"30-12-2022"),
(8,'prakash','ranchi',"24-02-2023"),
(9,'ragini','kolkata',"03-03-2023"),
(10,'raushan','jaipur',"05-02-2023")]

customer_schema=['customer_id','customer_name','address','date_of_joining']
customer_df = spark.createDataFrame(customer_data,customer_schema)


sales_data = [(1,22,10,"01-06-2022"),
(1,27,5,"03-02-2023"),
(2,5,3,"01-06-2023"),
(5,22,1,"22-03-2023"),
(7,22,4,"03-02-2023"),
(9,5,6,"03-03-2023"),
(2,1,12,"15-06-2023"),
(1,56,2,"25-06-2023"),
(5,12,5,"15-04-2023"),
(11,12,76,"12-03-2023")]

sales_schema=['customer_id','product_id','quantity','date_of_purchase']

sales_df = spark.createDataFrame(sales_data,sales_schema)

product_data = [(1, 'fanta',20),
(2, 'dew',22),
(5, 'sprite',40),
(7, 'redbull',100),
(12,'mazza',45),
(22,'coke',27),
(25,'limca',21),
(27,'pepsi',14),
(56,'sting',10)]

product_schema=['id','name','price']
product_df = spark.createDataFrame(product_data,product_schema)

In [0]:
customer_df.join(sales_df,sales_df['customer_id']==customer_df['customer_id'],"inner").show()

+-----------+-------------+-------+---------------+-----------+----------+--------+----------------+
|customer_id|customer_name|address|date_of_joining|customer_id|product_id|quantity|date_of_purchase|
+-----------+-------------+-------+---------------+-----------+----------+--------+----------------+
|          1|       manish|  patna|     30-05-2022|          1|        56|       2|      25-06-2023|
|          1|       manish|  patna|     30-05-2022|          1|        27|       5|      03-02-2023|
|          1|       manish|  patna|     30-05-2022|          1|        22|      10|      01-06-2022|
|          2|       vikash|kolkata|     12-03-2023|          2|         1|      12|      15-06-2023|
|          2|       vikash|kolkata|     12-03-2023|          2|         5|       3|      01-06-2023|
|          5|       mahesh| jaipur|     22-03-2023|          5|        12|       5|      15-04-2023|
|          5|       mahesh| jaipur|     22-03-2023|          5|        22|       1|      22

In [0]:
customer_df.join(sales_df,sales_df['customer_id']==customer_df['customer_id'],"inner").select('customer_id').show()

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-5827016565148875>, line 1
----> 1 customer_df.join(sales_df,sales_df['customer_id']==customer_df['customer_id'],"inner").select('customer_id').show()

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:1156, in DataFrame.show(self, n, truncate, vertical)
   1155 def show(self, n: int = 20, truncate: Union[bool, int] = True, vertical: bool = False) -> None:
-> 1156     print(self._show_string(n, truncate, vertical))

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:909, in DataFrame._show_string(self, n, truncate, vertical)
    892     except ValueError:
    893         raise PySparkTypeError(
    894             errorClass="NOT_BOOL",
    895             messageParameters={
   (...)
    898             },
    899         )
    901 table, _ = DataFrame(

In [0]:
customer_df.join(sales_df,sales_df['customer_id']==customer_df['customer_id'],"inner").select(sales_df['customer_id']).show()
# or we can use the alias column name

+-----------+
|customer_id|
+-----------+
|          1|
|          1|
|          1|
|          2|
|          2|
|          5|
|          5|
|          7|
|          9|
+-----------+



In [0]:
product_df.select("id").show()

+---+
| id|
+---+
|  1|
|  2|
|  5|
|  7|
| 12|
| 22|
| 25|
| 27|
| 56|
+---+



In [0]:
customer_df \
    .join(
        sales_df,
        customer_df["customer_id"] == sales_df["customer_id"],
        "inner"
    ) \
    .join(
        product_df,
        sales_df["product_id"] == product_df["id"],
        "inner"
    ) \
    .show()

+-----------+-------------+-------+---------------+-----------+----------+--------+----------------+---+------+-----+
|customer_id|customer_name|address|date_of_joining|customer_id|product_id|quantity|date_of_purchase| id|  name|price|
+-----------+-------------+-------+---------------+-----------+----------+--------+----------------+---+------+-----+
|          1|       manish|  patna|     30-05-2022|          1|        22|      10|      01-06-2022| 22|  coke|   27|
|          1|       manish|  patna|     30-05-2022|          1|        27|       5|      03-02-2023| 27| pepsi|   14|
|          2|       vikash|kolkata|     12-03-2023|          2|         5|       3|      01-06-2023|  5|sprite|   40|
|          5|       mahesh| jaipur|     22-03-2023|          5|        22|       1|      22-03-2023| 22|  coke|   27|
|          7|        raman|  patna|     30-12-2022|          7|        22|       4|      03-02-2023| 22|  coke|   27|
|          9|       ragini|kolkata|     03-03-2023|     

Type of joins

| Join Type        | Syntax (`how=`)                         | Returns                                                           |
| ---------------- | --------------------------------------- | ----------------------------------------------------------------- |
| Inner Join       | `"inner"` (default)                     | Matching rows from both DataFrames                                |
| Left Outer Join  | `"left"` or `"left_outer"`              | All rows from the left DataFrame + matching rows from the right   |
| Right Outer Join | `"right"` or `"right_outer"`            | All rows from the right DataFrame + matching rows from the left   |
| Full Outer Join  | `"outer"` or `"full"` or `"full_outer"` | All rows from both DataFrames                                     |
| Left Semi Join   | `"left_semi"`                           | Only matching rows from the left DataFrame (no right columns)     |
| Left Anti Join   | `"left_anti"`                           | Only non-matching rows from the left DataFrame                    |
| Cross Join       | `"cross"`                               | Cartesian product (every row from left with every row from right) |


In [0]:
# left outer join

customer_df.join(sales_df,sales_df['customer_id']==customer_df['customer_id'],"left").show()

+-----------+-------------+-------+---------------+-----------+----------+--------+----------------+
|customer_id|customer_name|address|date_of_joining|customer_id|product_id|quantity|date_of_purchase|
+-----------+-------------+-------+---------------+-----------+----------+--------+----------------+
|          1|       manish|  patna|     30-05-2022|          1|        56|       2|      25-06-2023|
|          1|       manish|  patna|     30-05-2022|          1|        27|       5|      03-02-2023|
|          1|       manish|  patna|     30-05-2022|          1|        22|      10|      01-06-2022|
|          2|       vikash|kolkata|     12-03-2023|          2|         1|      12|      15-06-2023|
|          2|       vikash|kolkata|     12-03-2023|          2|         5|       3|      01-06-2023|
|          3|       nikita|  delhi|     25-06-2023|       NULL|      NULL|    NULL|            NULL|
|          4|        rahul| ranchi|     24-03-2023|       NULL|      NULL|    NULL|        

In [0]:
# right outer join

sales_df.join(product_df,sales_df['product_id']==product_df['id'],"right").show()

+-----------+----------+--------+----------------+---+-------+-----+
|customer_id|product_id|quantity|date_of_purchase| id|   name|price|
+-----------+----------+--------+----------------+---+-------+-----+
|          2|         1|      12|      15-06-2023|  1|  fanta|   20|
|       NULL|      NULL|    NULL|            NULL|  2|    dew|   22|
|          9|         5|       6|      03-03-2023|  5| sprite|   40|
|          2|         5|       3|      01-06-2023|  5| sprite|   40|
|       NULL|      NULL|    NULL|            NULL|  7|redbull|  100|
|         11|        12|      76|      12-03-2023| 12|  mazza|   45|
|          5|        12|       5|      15-04-2023| 12|  mazza|   45|
|          7|        22|       4|      03-02-2023| 22|   coke|   27|
|          5|        22|       1|      22-03-2023| 22|   coke|   27|
|          1|        22|      10|      01-06-2022| 22|   coke|   27|
|       NULL|      NULL|    NULL|            NULL| 25|  limca|   21|
|          1|        27|       5| 

In [0]:
# full outer join

customer_df.join(sales_df,sales_df['customer_id']==customer_df['customer_id'],"full_outer").show()

+-----------+-------------+-------+---------------+-----------+----------+--------+----------------+
|customer_id|customer_name|address|date_of_joining|customer_id|product_id|quantity|date_of_purchase|
+-----------+-------------+-------+---------------+-----------+----------+--------+----------------+
|          1|       manish|  patna|     30-05-2022|          1|        56|       2|      25-06-2023|
|          2|       vikash|kolkata|     12-03-2023|          2|         1|      12|      15-06-2023|
|          3|       nikita|  delhi|     25-06-2023|       NULL|      NULL|    NULL|            NULL|
|          4|        rahul| ranchi|     24-03-2023|       NULL|      NULL|    NULL|            NULL|
|          5|       mahesh| jaipur|     22-03-2023|          5|        12|       5|      15-04-2023|
|          6|     prantosh|kolkata|     18-10-2022|       NULL|      NULL|    NULL|            NULL|
|          7|        raman|  patna|     30-12-2022|          7|        22|       4|      03

In [0]:
#left semi 

customer_df.join(sales_df,sales_df['customer_id']==customer_df['customer_id'],"left_semi").show()

+-----------+-------------+-------+---------------+
|customer_id|customer_name|address|date_of_joining|
+-----------+-------------+-------+---------------+
|          1|       manish|  patna|     30-05-2022|
|          2|       vikash|kolkata|     12-03-2023|
|          5|       mahesh| jaipur|     22-03-2023|
|          7|        raman|  patna|     30-12-2022|
|          9|       ragini|kolkata|     03-03-2023|
+-----------+-------------+-------+---------------+



In [0]:
#left anti join

customer_df.join(sales_df,sales_df['customer_id']==customer_df['customer_id'],"left_anti").show()

+-----------+-------------+-------+---------------+
|customer_id|customer_name|address|date_of_joining|
+-----------+-------------+-------+---------------+
|          3|       nikita|  delhi|     25-06-2023|
|          4|        rahul| ranchi|     24-03-2023|
|          6|     prantosh|kolkata|     18-10-2022|
|          8|      prakash| ranchi|     24-02-2023|
|         10|      raushan| jaipur|     05-02-2023|
+-----------+-------------+-------+---------------+



In [0]:
# cross join -- too much expensive (dont try to do this, try to ignore this)

customer_df.crossJoin(sales_df).show()

+-----------+-------------+-------+---------------+-----------+----------+--------+----------------+
|customer_id|customer_name|address|date_of_joining|customer_id|product_id|quantity|date_of_purchase|
+-----------+-------------+-------+---------------+-----------+----------+--------+----------------+
|          1|       manish|  patna|     30-05-2022|          1|        22|      10|      01-06-2022|
|          1|       manish|  patna|     30-05-2022|          1|        27|       5|      03-02-2023|
|          1|       manish|  patna|     30-05-2022|          2|         5|       3|      01-06-2023|
|          1|       manish|  patna|     30-05-2022|          5|        22|       1|      22-03-2023|
|          1|       manish|  patna|     30-05-2022|          7|        22|       4|      03-02-2023|
|          1|       manish|  patna|     30-05-2022|          9|         5|       6|      03-03-2023|
|          1|       manish|  patna|     30-05-2022|          2|         1|      12|      15

In [0]:
customer_df.crossJoin(sales_df).count()

100

Spark join short vs suffle

Optimization of joins

hash matching